In [11]:
from pathlib import Path
import pandas as pd

# 1. Definir rutas (ajusta el nombre si tu archivo se llama routes.dat)
ruta_dat = Path("raw_data") / "routes.dat"
ruta_csv = Path("data") / "rutas.csv"

# 2. Leer solo las columnas de ID origen (índice 3) e ID destino (índice 5)
# na_values="\\N" convierte los \N de OpenFlights en valores nulos reales (NaN)
df = pd.read_csv(
    ruta_dat,
    header=None,
    usecols=[3, 5],
    names=["id_aeropuerto_origen", "id_aeropuerto_destino"],
    na_values="\\N"
)

# 3. Limpiar datos para cumplir con las restricciones de tu tabla SQL:
# - Eliminar filas donde falte algún ID (NOT NULL)
df = df.dropna(subset=["id_aeropuerto_origen", "id_aeropuerto_destino"])

# - Convertir los IDs a enteros (al haber NaN antes, pandas los lee como float)
df["id_aeropuerto_origen"] = df["id_aeropuerto_origen"].astype(int)
df["id_aeropuerto_destino"] = df["id_aeropuerto_destino"].astype(int)

# - Eliminar rutas donde origen y destino sean el mismo (CHECK origen <> destino)
df = df[df["id_aeropuerto_origen"] != df["id_aeropuerto_destino"]]

# - Eliminar pares origen-destino duplicados (UNIQUE KEY uq_ruta)
df = df.drop_duplicates(subset=["id_aeropuerto_origen", "id_aeropuerto_destino"])

# 4. Añadir columnas restantes de tu tabla SQL
# id_ruta: autoincremental empezando en 1 (opcional si lo genera tu gestor SQL)
df.insert(0, "id_ruta", range(1, len(df) + 1))

# distancia_km: valor provisional > 0 (reemplazar por cálculo real si tienes coordenadas)
df["distancia_km"] = 1

# 5. Guardar a CSV sin el índice extra de pandas
df.to_csv(ruta_csv, index=False)

print(f"CSV guardado con {len(df)} rutas válidas.")
df.head()

CSV guardado con 37273 rutas válidas.


,id_ruta,id_aeropuerto_origen,id_aeropuerto_destino,distancia_km
0,1,2965,2990,1
1,2,2966,2990,1
2,3,2966,2962,1
3,4,2968,2990,1
4,5,2968,4078,1


In [12]:
from pathlib import Path
import pandas as pd

# 1. Definir rutas (ajusta el nombre si tu archivo se llama countries.dat)
ruta_dat = Path("raw_data") / "countries.dat"
ruta_csv = Path("data") / "paises.csv"

# 2. Leer columnas 0 (nombre) y 1 (codigo_iso)
# IMPORTANTE: keep_default_na=False evita que el código ISO de Namibia ("NA") se lea como nulo
df = pd.read_csv(
    ruta_dat,
    header=None,
    usecols=[0, 1],
    names=["nombre", "codigo_iso"],
    na_values=["\\N", ""],
    keep_default_na=False
)

# 3. Limpiar datos para cumplir con las restricciones SQL:
# - Eliminar filas donde falte codigo_iso o nombre (NOT NULL)
df = df.dropna(subset=["codigo_iso", "nombre"])

# - Limpiar espacios en blanco extra y asegurar mayúsculas en codigo_iso
df["codigo_iso"] = df["codigo_iso"].str.strip().str.upper()
df["nombre"] = df["nombre"].str.strip().str.slice(0, 50)

# - Eliminar códigos ISO duplicados (ej. IN o PS aparecen dos veces con distinto código DAFIF)
df = df.drop_duplicates(subset=["codigo_iso"], keep="first")

# - Ordenar columnas según tu CREATE TABLE: id_pais, codigo_iso, nombre
df = df[["codigo_iso", "nombre"]].reset_index(drop=True)

# 4. Añadir id_pais autoincremental (opcional si lo genera tu gestor SQL al importar)
df.insert(0, "id_pais", range(1, len(df) + 1))

# 5. Guardar a CSV
df.to_csv(ruta_csv, index=False)

print(f"CSV guardado con {len(df)} países únicos.")
df.head(10)

CSV guardado con 240 países únicos.


,id_pais,codigo_iso,nombre
0,1,BQ,"Bonaire, Saint Eustatius and Saba"
1,2,AW,Aruba
2,3,AG,Antigua and Barbuda
3,4,AE,United Arab Emirates
4,5,AF,Afghanistan
5,6,DZ,Algeria
6,7,AZ,Azerbaijan
7,8,AL,Albania
8,9,AM,Armenia
9,10,AO,Angola


In [15]:
from pathlib import Path
import pandas as pd

carpeta_raw = Path("raw_data")
carpeta_data = Path("data")
carpeta_data.mkdir(parents=True, exist_ok=True)

# ==========================================
# PASO 1: Generar data/pais.csv
# ==========================================
df_pais = pd.read_csv(
    carpeta_raw / "countries.dat",
    header=None,
    usecols=[0, 1],
    names=["nombre", "codigo_iso"],
    na_values=["\\N", ""],
    keep_default_na=False
)

df_pais = df_pais.dropna(subset=["codigo_iso", "nombre"]).copy()
df_pais["codigo_iso"] = df_pais["codigo_iso"].astype(str).str.strip().str.upper()
df_pais["nombre"] = df_pais["nombre"].astype(str).str.strip().str.slice(0, 50)
df_pais = df_pais.drop_duplicates(subset=["codigo_iso"], keep="first").reset_index(drop=True)
df_pais.insert(0, "id_pais", range(1, len(df_pais) + 1))
df_pais = df_pais[["id_pais", "codigo_iso", "nombre"]]

df_pais.to_csv(carpeta_data / "pais.csv", index=False)
print(f"1/2 -> Creado data/pais.csv ({len(df_pais)} países)")

# ==========================================
# PASO 2: Generar data/aeropuerto.csv
# ==========================================
df_aero = pd.read_csv(
    carpeta_raw / "airports.dat",
    header=None,
    usecols=[0, 1, 2, 3, 4, 5, 6, 7, 11],
    names=[
        "id_aeropuerto",
        "nombre",
        "ciudad",
        "pais_nombre",
        "codigo_iata",
        "codigo_icao",
        "latitud",
        "longitud",
        "zona_horaria"
    ],
    na_values=["\\N", ""],
    keep_default_na=False
)

# Cruzar con df_pais para obtener id_pais (FOREIGN KEY)
df_aero = df_aero.merge(
    df_pais[["id_pais", "nombre"]],
    left_on="pais_nombre",
    right_on="nombre",
    how="inner",
    suffixes=("", "_pais")
)

# Limpiar nulos y duplicados según restricciones SQL
columnas_obligatorias = [
    "id_aeropuerto", "codigo_iata", "codigo_icao",
    "nombre", "ciudad", "id_pais", "latitud", "longitud"
]
df_aero = df_aero.dropna(subset=columnas_obligatorias).copy()

for col in ["codigo_iata", "codigo_icao", "nombre", "ciudad"]:
    df_aero[col] = df_aero[col].astype(str).str.strip()

df_aero["codigo_iata"] = df_aero["codigo_iata"].str.upper()
df_aero["codigo_icao"] = df_aero["codigo_icao"].str.upper()

# Solo códigos válidos de 3 (IATA) y 4 (ICAO) caracteres
df_aero = df_aero[
    (df_aero["codigo_iata"].str.len() == 3) &
    (df_aero["codigo_icao"].str.len() == 4)
]

# Evitar duplicados en UNIQUE KEY
df_aero = df_aero.drop_duplicates(subset=["codigo_iata"], keep="first")
df_aero = df_aero.drop_duplicates(subset=["codigo_icao"], keep="first")

# Ajustar longitud VARCHAR(50) y tipos numéricos
df_aero["nombre"] = df_aero["nombre"].str.slice(0, 50)
df_aero["ciudad"] = df_aero["ciudad"].str.slice(0, 50)
df_aero["zona_horaria"] = df_aero["zona_horaria"].apply(
    lambda x: str(x).strip()[:50] if pd.notna(x) and str(x).strip() != "" else None
)

df_aero["id_aeropuerto"] = pd.to_numeric(df_aero["id_aeropuerto"], errors="coerce").astype(int)
df_aero["id_pais"] = df_aero["id_pais"].astype(int)
df_aero["latitud"] = pd.to_numeric(df_aero["latitud"], errors="coerce")
df_aero["longitud"] = pd.to_numeric(df_aero["longitud"], errors="coerce")
df_aero = df_aero.dropna(subset=["latitud", "longitud"])

columnas_finales = [
    "id_aeropuerto", "codigo_iata", "codigo_icao",
    "nombre", "ciudad", "id_pais",
    "latitud", "longitud", "zona_horaria"
]
df_aero = df_aero[columnas_finales]

df_aero.to_csv(carpeta_data / "aeropuerto.csv", index=False)
print(f"2/2 -> Creado data/aeropuerto.csv ({len(df_aero)} aeropuertos)")
df_aero.head()

1/2 -> Creado data/pais.csv (240 países)
2/2 -> Creado data/aeropuerto.csv (5906 aeropuertos)


,id_aeropuerto,codigo_iata,codigo_icao,nombre,ciudad,id_pais,latitud,longitud,zona_horaria
0,1,GKA,AYGA,Goroka Airport,Goroka,174,-6.081690,145.391998,Pacific/Port_Moresby
1,2,MAG,AYMD,Madang Airport,Madang,174,-5.207080,145.789001,Pacific/Port_Moresby
2,3,HGU,AYMH,Mount Hagen Kagamuga Airport,Mount Hagen,174,-5.826790,144.296005,Pacific/Port_Moresby
3,4,LAE,AYNZ,Nadzab Airport,Nadzab,174,-6.569803,146.725977,Pacific/Port_Moresby
4,5,POM,AYPY,Port Moresby Jacksons International Airport,Port Moresby,174,-9.443380,147.220001,Pacific/Port_Moresby


In [14]:
from pathlib import Path
import pandas as pd

# 1. Definir rutas
carpeta_raw = Path("raw_data")
carpeta_data = Path("data")
carpeta_data.mkdir(parents=True, exist_ok=True)  # Crea la carpeta /data si no existe

ruta_airports_dat = carpeta_raw / "airports.dat"
ruta_pais_csv = carpeta_data / "pais.csv"
ruta_aeropuerto_csv = carpeta_data / "aeropuerto.csv"

# 2. Leer las columnas necesarias de airports.dat
# Índices: 0=id, 1=nombre, 2=ciudad, 3=pais, 4=iata, 5=icao, 6=latitud, 7=longitud, 11=zona_horaria
df_aero = pd.read_csv(
    ruta_airports_dat,
    header=None,
    usecols=[0, 1, 2, 3, 4, 5, 6, 7, 11],
    names=[
        "id_aeropuerto",
        "nombre",
        "ciudad",
        "pais_nombre",
        "codigo_iata",
        "codigo_icao",
        "latitud",
        "longitud",
        "zona_horaria"
    ],
    na_values=["\\N", ""],
    keep_default_na=False
)

# 3. Cargar data/pais.csv para mapear el nombre del país a su id_pais (FOREIGN KEY)
df_pais = pd.read_csv(ruta_pais_csv, keep_default_na=False)

# Unimos por nombre de país (inner join asegura que todo id_pais exista en la tabla pais)
df_aero = df_aero.merge(
    df_pais[["id_pais", "nombre"]],
    left_on="pais_nombre",
    right_on="nombre",
    how="inner",
    suffixes=("", "_pais")
)

# 4. Limpiar datos según las restricciones de la tabla aeropuerto:
# - Eliminar filas con nulos en columnas NOT NULL
columnas_obligatorias = [
    "id_aeropuerto", "codigo_iata", "codigo_icao",
    "nombre", "ciudad", "id_pais", "latitud", "longitud"
]
df_aero = df_aero.dropna(subset=columnas_obligatorias)

# - Limpiar espacios y validar longitud exacta de IATA (3) e ICAO (4)
df_aero["codigo_iata"] = df_aero["codigo_iata"].str.strip().str.upper()
df_aero["codigo_icao"] = df_aero["codigo_icao"].str.strip().str.upper()
df_aero = df_aero[
    (df_aero["codigo_iata"].str.len() == 3) &
    (df_aero["codigo_icao"].str.len() == 4)
]

# - Eliminar duplicados para cumplir con UNIQUE KEY en codigo_iata y codigo_icao
df_aero = df_aero.drop_duplicates(subset=["codigo_iata"], keep="first")
df_aero = df_aero.drop_duplicates(subset=["codigo_icao"], keep="first")

# - Ajustar longitudes máximas a VARCHAR(50) y asegurar tipos numéricos
df_aero["nombre"] = df_aero["nombre"].str.strip().str.slice(0, 50)
df_aero["ciudad"] = df_aero["ciudad"].str.strip().str.slice(0, 50)
df_aero["zona_horaria"] = df_aero["zona_horaria"].str.strip().str.slice(0, 50)
df_aero["id_aeropuerto"] = df_aero["id_aeropuerto"].astype(int)
df_aero["id_pais"] = df_aero["id_pais"].astype(int)

# 5. Ordenar las columnas exactamente como en el CREATE TABLE
columnas_finales = [
    "id_aeropuerto",
    "codigo_iata",
    "codigo_icao",
    "nombre",
    "ciudad",
    "id_pais",
    "latitud",
    "longitud",
    "zona_horaria"
]
df_aero = df_aero[columnas_finales]

# 6. Exportar a data/aeropuerto.csv
df_aero.to_csv(ruta_aeropuerto_csv, index=False)

print(f"Archivo guardado en {ruta_aeropuerto_csv} con {len(df_aero)} aeropuertos válidos.")
df_aero.head()

FileNotFoundError: [Errno 2] No such file or directory: 'data\\pais.csv'

In [1]:
from pathlib import Path
import re
import itertools
import unicodedata
import pandas as pd

carpeta_raw = Path("raw_data")
carpeta_data = Path("data")
carpeta_data.mkdir(parents=True, exist_ok=True)

ruta_airlines_dat = carpeta_raw / "airlines.dat"
ruta_pais_csv = carpeta_data / "pais.csv"
ruta_aerolinea_csv = carpeta_data / "aerolinea.csv"

# ---------------------------------------------------------------------------
# CONFIGURACIÓN
# ---------------------------------------------------------------------------
# Un código IATA de aerolínea tiene 2 caracteres alfanuméricos: como máximo
# 36 * 36 = 1.296 códigos distintos. airlines.dat tiene casi 6.000 aerolíneas,
# así que es imposible que todas tengan un IATA único. Por eso:
#   - las aerolíneas ACTIVAS tienen prioridad absoluta: primero conservan su
#     código real y después se les genera uno si no tienen;
#   - a las INACTIVAS solo se les respeta su código real si sigue libre, y
#     nunca se les genera uno nuevo.
ELIMINAR_SIN_IATA = True   # True: el CSV se puede cargar en un CHAR(2) UNIQUE NOT NULL.
                           # False: se conservan todas (habría que permitir NULL en la columna).
ALFABETO = "ABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789"
MAX_CODIGOS = len(ALFABETO) ** 2  # 1.296

# Palabras que no distinguen a una aerolínea y que no aportan a su código.
PALABRAS_GENERICAS = {
    "AIR", "AIRS", "AIRLINE", "AIRLINES", "AIRWAYS", "AIRWAY", "AVIATION", "AIRCRAFT",
    "AIRLINK", "LTD", "LIMITED", "INC", "CO", "CORP", "CORPORATION", "COMPANY", "SA",
    "SL", "THE", "OF", "AND", "DE", "DEL", "LA", "EL", "Y",
}


# ---------------------------------------------------------------------------
# FUNCIONES
# ---------------------------------------------------------------------------
def limpiar_codigo(x, longitud: int):
    """Mayúsculas y solo A-Z/0-9. Devuelve None si falta o no tiene la longitud pedida."""
    if pd.isna(x):
        return None
    codigo = re.sub(r"[^A-Z0-9]", "", str(x).strip().upper())
    return codigo if len(codigo) == longitud else None


def normalizar_nombre(nombre: str) -> str:
    """Quita tildes y símbolos: "Afric'air Express" -> "AFRIC AIR EXPRESS"."""
    sin_tildes = unicodedata.normalize("NFKD", nombre).encode("ascii", "ignore").decode()
    return re.sub(r"[^A-Z0-9 ]", " ", sin_tildes.upper())


def candidatos_por_nombre(nombre: str) -> list:
    """Códigos de 2 caracteres derivados del nombre, de más a menos 'con sentido'."""
    palabras = normalizar_nombre(nombre).split()
    significativas = [p for p in palabras if p not in PALABRAS_GENERICAS] or palabras
    if not significativas:
        return []

    letras = "".join(significativas)      # "Air Southwest" -> "SOUTHWEST"
    inicial = letras[0]
    candidatos = []

    if len(significativas) >= 2:          # iniciales de las dos primeras palabras
        candidatos.append(inicial + significativas[1][0])
    candidatos.append(letras[:2])         # dos primeras letras
    if len(significativas) >= 2:          # inicial de la primera + inicial de la última
        candidatos.append(inicial + significativas[-1][0])

    resto = letras[1:]                    # inicial + cada letra siguiente (consonantes primero)
    consonantes = [c for c in resto if c.isalpha() and c not in "AEIOU"]
    otras = [c for c in resto if c.isalnum()]
    for c in consonantes + otras:
        candidatos.append(inicial + c)
    return candidatos


def generar_codigo_iata(nombre: str, icao, usados: set, evitar=frozenset()):
    """Genera un código IATA de 2 caracteres a partir del NOMBRE de la aerolínea.

    Se llama solo cuando el codigo_iata es nulo o está vacío. Orden de intentos:
      1. Combinaciones del nombre (iniciales, primeras letras, consonantes...).
      2. Combinaciones del código ICAO (solo como apoyo).
      3. La inicial del nombre combinada con cualquier carácter (inicial+X, X+inicial).
      4. Último recurso: barrido por todo el espacio (los códigos solo de dígitos, al final).
    Un código con sentido (pasos 1 a 3) siempre gana a uno arbitrario, aunque haya que
    quitarle su código real a una aerolínea inactiva (`evitar`): a las inactivas solo
    se les respeta el código cuando no estorba. Registra el elegido en `usados` y lo
    devuelve; devuelve None si se agotan los 1.296.
    """
    if len(usados) >= MAX_CODIGOS:
        return None

    con_sentido = candidatos_por_nombre(nombre)
    if isinstance(icao, str) and len(icao) == 3:
        con_sentido += [icao[:2], icao[0] + icao[2], icao[1:]]
    inicial = next((c for c in normalizar_nombre(nombre) if c.isalpha()), "X")
    con_sentido += [inicial + c for c in ALFABETO] + [c + inicial for c in ALFABETO]

    barrido = ["".join(par) for par in itertools.product(ALFABETO, repeat=2)]
    arbitrarios = [c for c in barrido if not c.isdigit()] + [c for c in barrido if c.isdigit()]

    for candidatos in (con_sentido, arbitrarios):
        for excluidos in (usados | evitar, usados):  # 1º respetando a las inactivas, 2º sin respetarlas
            for candidato in candidatos:
                if len(candidato) == 2 and candidato not in excluidos:
                    usados.add(candidato)
                    return candidato
    return None


# ---------------------------------------------------------------------------
# 1. Leer airlines.dat
# ---------------------------------------------------------------------------
df_air = pd.read_csv(
    ruta_airlines_dat,
    header=None,
    usecols=[0, 1, 3, 4, 5, 6, 7],
    names=[
        "id_aerolinea",
        "nombre",
        "codigo_iata",
        "codigo_icao",
        "indicativo",
        "pais_nombre",
        "activa"
    ],
    na_values=["\\N", "", "-", "N/A"],
    keep_default_na=False,
    on_bad_lines="skip"
)

# 2. Cruzar con pais.csv
df_pais = pd.read_csv(ruta_pais_csv, keep_default_na=False)
equivalencias_pais = {
    "Republic of Korea": "South Korea",
    "Democratic Republic of the Congo": "DR Congo",
    "Ivory Coast": "Cote d'Ivoire",
    "Somali Republic": "Somalia",
    "Swaziland": "Eswatini"
}
df_air["pais_nombre"] = df_air["pais_nombre"].replace(equivalencias_pais)

df_air = df_air.merge(
    df_pais[["id_pais", "nombre"]],
    left_on="pais_nombre",
    right_on="nombre",
    how="inner",
    suffixes=("", "_pais")
)

# 3. Limpieza base
df_air = df_air.dropna(subset=["id_aerolinea", "nombre", "id_pais", "activa"]).copy()
df_air["id_aerolinea"] = pd.to_numeric(df_air["id_aerolinea"], errors="coerce")
df_air = df_air[df_air["id_aerolinea"] > 0].copy()
df_air["id_aerolinea"] = df_air["id_aerolinea"].astype(int)

df_air["nombre"] = df_air["nombre"].astype(str).str.strip().str.slice(0, 50)
df_air["activa"] = df_air["activa"].astype(str).str.strip().str.upper()
df_air = df_air[df_air["activa"].isin(["Y", "N"])]
df_air["activa"] = (df_air["activa"] == "Y").astype(int)

# IATA de 2 caracteres e ICAO de 3; lo que no cumpla queda en None (= "nulo o vacío")
df_air["codigo_iata"] = df_air["codigo_iata"].apply(lambda x: limpiar_codigo(x, 2))
df_air["codigo_icao"] = df_air["codigo_icao"].apply(lambda x: limpiar_codigo(x, 3))

df_air["indicativo"] = df_air["indicativo"].apply(
    lambda x: str(x).strip()[:50] if pd.notna(x) and str(x).strip() != "" else None
)

df_air = df_air.drop_duplicates(subset=["id_aerolinea"], keep="first")

# Orden de prioridad: activas primero y, dentro de ellas, las que tienen ICAO
df_air = df_air.sort_values(
    by=["activa", "codigo_icao"],
    ascending=[False, True],
    na_position="last"
).reset_index(drop=True)

# ---------------------------------------------------------------------------
# 4. Asignar códigos IATA por prioridad
# ---------------------------------------------------------------------------
usados_iata = set()
df_air["origen_iata"] = None  # "real" o "generado" (columna auxiliar, no se exporta)
activas = df_air["activa"] == 1


def registrar_codigos_reales(mascara):
    """Conserva el IATA real de quien llega antes; si ya está usado, lo deja en None."""
    for idx in df_air[mascara].index:
        iata = df_air.at[idx, "codigo_iata"]
        if pd.isna(iata):
            continue
        if iata in usados_iata:
            df_air.at[idx, "codigo_iata"] = None
        else:
            usados_iata.add(iata)
            df_air.at[idx, "origen_iata"] = "real"


# 4a. Códigos reales de las aerolíneas activas
registrar_codigos_reales(activas)

# 4b. Códigos reales de las inactivas que conviene no pisar al generar
reservados_inactivas = {
    c for c in df_air.loc[~activas, "codigo_iata"].dropna() if c not in usados_iata
}

# 4c. Generar el código (a partir del nombre) de las activas con IATA nulo o vacío
for idx in df_air[activas & df_air["codigo_iata"].isna()].index:
    nuevo = generar_codigo_iata(
        df_air.at[idx, "nombre"], df_air.at[idx, "codigo_icao"], usados_iata, reservados_inactivas
    )
    df_air.at[idx, "codigo_iata"] = nuevo
    if nuevo is not None:
        df_air.at[idx, "origen_iata"] = "generado"

# 4d. Las inactivas conservan su código real solo si sigue libre (a ellas no se les genera)
registrar_codigos_reales(~activas)

# 5. Quedarse solo con las aerolíneas que tienen IATA (si el SQL exige NOT NULL)
sin_iata = int(df_air["codigo_iata"].isna().sum())
if ELIMINAR_SIN_IATA:
    df_air = df_air.dropna(subset=["codigo_iata"])

assert df_air["codigo_iata"].dropna().is_unique, "IATA duplicados"

# 6. Ordenar por id_aerolinea y exportar
df_air = df_air.sort_values("id_aerolinea").reset_index(drop=True)
columnas_finales = [
    "id_aerolinea",
    "nombre",
    "codigo_iata",
    "codigo_icao",
    "indicativo",
    "id_pais",
    "activa"
]
resumen = df_air["origen_iata"].value_counts().to_dict()
df_air[columnas_finales].to_csv(ruta_aerolinea_csv, index=False)

print(
    f"CSV guardado con {len(df_air)} aerolíneas | IATA reales: {resumen.get('real', 0)}, "
    f"generados: {resumen.get('generado', 0)} | aerolíneas sin IATA: {sin_iata} "
    f"({'eliminadas' if ELIMINAR_SIN_IATA else 'conservadas con IATA vacío'})."
)
solo_digitos = int(((df_air["origen_iata"] == "generado") & df_air["codigo_iata"].str.isdigit()).sum())
if solo_digitos:
    print(f"Nota: {solo_digitos} códigos generados son solo dígitos (se agotaron los que llevan letras).")
activas_sin_iata = int(((df_air["activa"] == 1) & df_air["codigo_iata"].isna()).sum())
if activas_sin_iata:
    print(f"AVISO: {activas_sin_iata} aerolíneas activas se han quedado sin IATA (se agotaron los 1.296).")

df_air[columnas_finales].head(15)

CSV guardado con 1245 aerolíneas | IATA reales: 984, generados: 261 | aerolíneas sin IATA: 4738 (eliminadas).
Nota: 32 códigos generados son solo dígitos (se agotaron los que llevan letras).


,id_aerolinea,nombre,codigo_iata,codigo_icao,indicativo,id_pais,activa
0,3,1Time Airline,1T,RNX,NEXTIME,190,1
1,10,40-Mile Air,Q5,MLA,MILE-AIR,224,1
2,13,Ansett Australia,AN,AAA,ANSETT,13,1
3,14,Abacus International,1B,NaN,NaN,195,1
4,21,Aigle Azur,ZI,AAF,AIGLE AZUR,78,1
5,22,Aloha Airlines,AQ,AAH,ALOHA,224,1
6,24,American Airlines,AA,AAL,AMERICAN,224,1
7,28,Asiana Airlines,OZ,AAR,ASIANA,119,1
8,29,Askari Aviation,4K,AAS,AL-AAS,170,1
9,32,Afriqiyah Airways,8U,AAW,AFRIQIYAH,131,1


In [3]:
from pathlib import Path
import pandas as pd

# 1. Definir rutas
carpeta_raw = Path("raw_data")
carpeta_data = Path("data")
carpeta_data.mkdir(parents=True, exist_ok=True)

ruta_runways_dat = carpeta_raw / "runways.csv"
ruta_aeropuerto_csv = carpeta_data / "aeropuerto.csv"
ruta_pista_csv = carpeta_data / "pista.csv"

# Comprobación de que existe aeropuerto.csv para la integridad referencial
if not ruta_aeropuerto_csv.exists():
    raise FileNotFoundError(
        "Falta 'aeropuerto.csv' en la carpeta data/. Genéralo antes de procesar las pistas."
    )

# 2. Leer runways.csv
df_pistas = pd.read_csv(
    ruta_runways_dat,
    usecols=[
        "id",
        "airport_ref",
        "length_ft",
        "le_ident",
        "he_ident"
    ],
    na_values=["\\N", ""]
)

# 3. Cargar aeropuertos válidos para asegurar la FK
df_aero = pd.read_csv(ruta_aeropuerto_csv)
ids_aeropuertos_validos = set(df_aero["id_aeropuerto"])

# 4. Limpieza y transformaciones
# - Filtrar aeropuertos que existan en la tabla aeropuerto
df_pistas = df_pistas[df_pistas["airport_ref"].isin(ids_aeropuertos_validos)].copy()

# - Eliminar filas con nulos en ID o longitud
df_pistas = df_pistas.dropna(subset=["id", "airport_ref", "length_ft"])

# - Construir la designación combinando le_ident y he_ident (ej. "04/22")
def crear_designacion(row):
    le = str(row["le_ident"]).strip() if pd.notna(row["le_ident"]) else ""
    he = str(row["he_ident"]).strip() if pd.notna(row["he_ident"]) else ""
    
    if le and he and le.lower() != "nan" and he.lower() != "nan":
        return f"{le}/{he}"
    elif le and le.lower() != "nan":
        return le
    elif he and he.lower() != "nan":
        return he
    return "UNKNOWN"

df_pistas["designacion"] = df_pistas.apply(crear_designacion, axis=1)

# - Convertir longitud de pies a metros (1 ft = 0.3048 m) y asegurar INT > 0
df_pistas["longitud_m"] = (df_pistas["length_ft"] * 0.3048).round().astype(int)
df_pistas = df_pistas[df_pistas["longitud_m"] > 0]

# - Renombrar columnas para que coincidan exactamente con el CREATE TABLE
df_pistas = df_pistas.rename(columns={
    "id": "id_pista",
    "airport_ref": "id_aeropuerto"
})

# - Seleccionar y ordenar columnas finales
columnas_finales = [
    "id_pista",
    "id_aeropuerto",
    "designacion",
    "longitud_m"
]
df_pistas = df_pistas[columnas_finales]

# 5. Exportar a CSV
df_pistas.to_csv(ruta_pista_csv, index=False)

print(f"Archivo guardado en {ruta_pista_csv} con {len(df_pistas)} pistas válidas.")
df_pistas.head(10)

Archivo guardado en data\pista.csv con 7276 pistas válidas.


,id_pista,id_aeropuerto,designacion,longitud_m
187,576793,6712,H1,8
188,250552,6713,03/21,701
189,265072,6714,H1,18
190,265073,6715,H1,12
191,258174,6716,02/20,1158
192,256543,6717,12/30,488
193,265074,6718,H1,15
194,251983,6719,18/36,732
195,265075,6720,H1,6
197,265076,6723,H1,18


In [6]:
from pathlib import Path
import shutil

import pandas as pd

# 1. Definir rutas
carpeta_raw = Path("raw_data")
carpeta_data = Path("data")
carpeta_raw.mkdir(parents=True, exist_ok=True)

ruta_terminal_csv = carpeta_data / "terminal.csv"
ruta_aeropuerto_csv = carpeta_data / "aeropuerto.csv"
ruta_copia = carpeta_raw / "terminal_original.csv"

for ruta in (ruta_terminal_csv, ruta_aeropuerto_csv):
    if not ruta.exists():
        raise FileNotFoundError(f"Falta '{ruta}'.")

# 2. Copia de seguridad del terminal.csv original (solo la primera vez)
if not ruta_copia.exists():
    shutil.copy(ruta_terminal_csv, ruta_copia)

# 3. Leer siempre desde la copia original, así el script se puede repetir
df_term = pd.read_csv(ruta_copia, dtype=str)
df_aero = pd.read_csv(ruta_aeropuerto_csv, dtype={"codigo_iata": str})

for col in ("codigo_iata", "nombre"):
    if col not in df_term.columns:
        raise KeyError(f"terminal_original.csv necesita la columna '{col}'. Tiene: {list(df_term.columns)}")

# 4. Limpieza
df_term["codigo_iata"] = df_term["codigo_iata"].str.strip().str.upper()
df_term["nombre"] = df_term["nombre"].str.strip()
df_term = df_term.dropna(subset=["codigo_iata", "nombre"])
df_term = df_term[df_term["nombre"] != ""]
df_term["nombre"] = df_term["nombre"].str.slice(0, 50)  # VARCHAR(50)

# 5. Cruzar con aeropuerto para obtener id_aeropuerto (FK)
df_aero["codigo_iata"] = df_aero["codigo_iata"].str.strip().str.upper()
df = df_term.merge(df_aero[["codigo_iata", "id_aeropuerto"]], on="codigo_iata", how="left")

sin_aeropuerto = df[df["id_aeropuerto"].isna()]
df = df.dropna(subset=["id_aeropuerto"]).copy()
df["id_aeropuerto"] = df["id_aeropuerto"].astype(int)

# 6. Quitar duplicados y crear id_terminal
df = df.drop_duplicates(subset=["id_aeropuerto", "nombre"])
df = df.sort_values(["id_aeropuerto", "nombre"]).reset_index(drop=True)
df["id_terminal"] = df.index + 1

# 7. Columnas finales, iguales al CREATE TABLE
df_final = df[["id_terminal", "id_aeropuerto", "nombre"]]
df_final.to_csv(ruta_terminal_csv, index=False)

print(f"Terminales leídos: {len(df_term)}")
print(f"Descartados por no estar su aeropuerto en aeropuerto.csv: {len(sin_aeropuerto)}")
if len(sin_aeropuerto):
    print(sorted(sin_aeropuerto["codigo_iata"].unique()))
print(f"Archivo guardado en {ruta_terminal_csv} con {len(df_final)} terminales.")
df_final.head(10)

Terminales leídos: 313
Descartados por no estar su aeropuerto en aeropuerto.csv: 1
['RAI']
Archivo guardado en data\terminal.csv con 312 terminales.


,id_terminal,id_aeropuerto,nombre
0,1,16,Terminal 1
1,2,146,Terminal 1
2,3,193,Terminal 1
3,4,210,Terminal 1
4,5,231,Terminal 1
5,6,248,Terminal 1
6,7,273,Terminal 1
7,8,287,Terminal 1
8,9,293,Terminal 1
9,10,302,Terminal 1


In [7]:
from pathlib import Path
import json
import re

import numpy as np
import pandas as pd

# 1. Definir rutas
carpeta_raw = Path("raw_data")
carpeta_data = Path("data")
carpeta_data.mkdir(parents=True, exist_ok=True)

ruta_geojson = carpeta_raw / "puerta_embarque.geojson"
ruta_aeropuerto_csv = carpeta_data / "aeropuerto.csv"
ruta_terminal_csv = carpeta_data / "terminal.csv"
ruta_puerta_csv = carpeta_data / "puerta_embarque.csv"

for ruta in (ruta_geojson, ruta_aeropuerto_csv, ruta_terminal_csv):
    if not ruta.exists():
        raise FileNotFoundError(f"Falta '{ruta}'. Genéralo antes de procesar las puertas.")

# Radio máximo (km) para asignar una puerta al aeropuerto más cercano
RADIO_MAX_KM = 6.0

# Aeropuertos con varios terminales: letra inicial de la puerta -> terminal.
# Se aplica solo si el aeropuerto tiene más de un terminal en terminal.csv.
# Deduplicado por geografía en el GeoJSON de MAD: H/J/K (oeste) = T4; R/S/U (este) = T4S.
# Las letras A-E (T1/T2/T3) no se asignan: rellénalas tú tras ver el informe final.
PREFIJO_TERMINAL = {
    "MAD": {"H": "T4", "J": "T4", "K": "T4", "R": "T4S", "S": "T4S", "U": "T4S"},
}

# 2. Leer GeoJSON (solo nodos aeroway=gate)
with open(ruta_geojson, encoding="utf-8") as f:
    geo = json.load(f)

filas = []
for feat in geo["features"]:
    props = feat.get("properties", {})
    geom = feat.get("geometry", {})
    if props.get("aeroway") != "gate" or geom.get("type") != "Point":
        continue
    lon, lat = geom["coordinates"][:2]
    filas.append({
        "osm_id": props.get("@id"),
        "texto": props.get("ref") or props.get("name"),
        "lat": lat,
        "lon": lon,
    })
df = pd.DataFrame(filas)
n_inicial = len(df)

# 3. Limpiar el código de la puerta
df = df.dropna(subset=["texto"]).copy()
df["texto"] = df["texto"].astype(str).str.strip().str.upper()

# - Descartar referencias solo numéricas (suelen ser stands de plataforma, no puertas)
df = df[df["texto"].str.contains(r"^[A-Z]")]


def expandir(texto):
    """'H16/H18' -> [H16, H18]; 'B18-B20' -> [B18, B19, B20]; 'A1' -> [A1]."""
    codigos = []
    if re.search(r"[A-Z]/[A-Z]\d", texto):  # 'B/C30-B/C33': ambiguo, se descarta
        return codigos
    for parte in re.split(r"[;/,]\s*(?=[A-Z]?\d)", texto):
        parte = parte.strip()
        m = re.fullmatch(r"([A-Z]+)(\d+)\s*-\s*\1?(\d+)", parte)
        if m:
            letra, ini, fin = m.group(1), int(m.group(2)), int(m.group(3))
            if 0 <= fin - ini <= 30:
                codigos += [f"{letra}{n}" for n in range(ini, fin + 1)]
                continue
        if re.fullmatch(r"[A-Z]+\d+[A-Z]?", parte):
            codigos.append(parte)
    return codigos


df["codigo"] = df["texto"].apply(expandir)
n_sin_codigo = int((df["codigo"].str.len() == 0).sum())
df = df.explode("codigo").dropna(subset=["codigo"])
df = df[df["codigo"].str.len() <= 10]

# 4. Asignar aeropuerto más cercano (Haversine)
df_aero = pd.read_csv(ruta_aeropuerto_csv)
a_lat = np.radians(df_aero["latitud"].to_numpy())
a_lon = np.radians(df_aero["longitud"].to_numpy())


def aeropuerto_cercano(lat, lon):
    la, lo = np.radians(lat), np.radians(lon)
    h = np.sin((a_lat - la) / 2) ** 2 + np.cos(la) * np.cos(a_lat) * np.sin((a_lon - lo) / 2) ** 2
    d = 2 * 6371 * np.arcsin(np.sqrt(h))
    i = int(d.argmin())
    return (df_aero["id_aeropuerto"].iloc[i], df_aero["codigo_iata"].iloc[i]) if d[i] <= RADIO_MAX_KM else (None, None)


res = [aeropuerto_cercano(la, lo) for la, lo in zip(df["lat"], df["lon"])]
df["id_aeropuerto"] = [r[0] for r in res]
df["iata"] = [r[1] for r in res]
df = df.dropna(subset=["id_aeropuerto"]).copy()
df["id_aeropuerto"] = df["id_aeropuerto"].astype(int)

# 5. Asignar terminal
df_term = pd.read_csv(ruta_terminal_csv)  # columnas esperadas: id_terminal, id_aeropuerto, nombre


def norm(nombre):
    s = re.sub(r"TERMINAL|\s", "", str(nombre).upper())
    return re.sub(r"^T(?=\d)", "", s)


df_term["clave"] = df_term["nombre"].apply(norm)
terminales_por_aero = {k: g for k, g in df_term.groupby("id_aeropuerto")}


def asignar_terminal(row):
    g = terminales_por_aero.get(row["id_aeropuerto"])
    if g is None:
        return None
    if len(g) == 1:
        return g["id_terminal"].iloc[0]
    destino = PREFIJO_TERMINAL.get(row["iata"], {}).get(row["codigo"][0])
    if destino is None:
        return None
    m = g[g["clave"] == norm(destino)]
    return m["id_terminal"].iloc[0] if len(m) else None


df["id_terminal"] = df.apply(asignar_terminal, axis=1)
sin_terminal = df[df["id_terminal"].isna()]
df = df.dropna(subset=["id_terminal"]).copy()
df["id_terminal"] = df["id_terminal"].astype(int)

# 6. Deduplicar por la UNIQUE (id_terminal, codigo) y crear id_puerta
df = df.drop_duplicates(subset=["id_terminal", "codigo"]).sort_values(["id_terminal", "codigo"])
df = df.reset_index(drop=True)
df["id_puerta"] = df.index + 1

df_final = df[["id_puerta", "id_terminal", "codigo"]]

# 7. Exportar
df_final.to_csv(ruta_puerta_csv, index=False)

print(f"Nodos gate leídos: {n_inicial}")
print(f"Sin código válido (p. ej. 'B/C30-B/C33'): {n_sin_codigo}")
print(f"Sin terminal asignado: {len(sin_terminal)}")
if len(sin_terminal):
    print(sin_terminal.assign(letra=sin_terminal["codigo"].str[0]).groupby(["iata", "letra"]).size().to_string())
print(f"Archivo guardado en {ruta_puerta_csv} con {len(df_final)} puertas válidas.")
df_final.head(10)

Nodos gate leídos: 104
Sin código válido (p. ej. 'B/C30-B/C33'): 1
Sin terminal asignado: 73
iata  letra
MAD   A        14
      B        12
      C        17
      D        14
      E        16
Archivo guardado en data\puerta_embarque.csv con 124 puertas válidas.


,id_puerta,id_terminal,codigo
0,1,152,H1
1,2,152,H11
2,3,152,H13
3,4,152,H15
4,5,152,H16
5,6,152,H18
6,7,152,H19
7,8,152,H2
8,9,152,H21
9,10,152,H22


In [8]:
from pathlib import Path

import numpy as np
import pandas as pd

# 1. Definir rutas
carpeta_raw = Path("raw_data")
carpeta_data = Path("data")
carpeta_data.mkdir(parents=True, exist_ok=True)

ruta_opensky = carpeta_raw / "aircraft-database-complete-2025-08.csv"
ruta_modelo_csv = carpeta_data / "modelo_aeronave.csv"
ruta_aerolinea_csv = carpeta_data / "aerolinea.csv"
ruta_aeronave_csv = carpeta_data / "aeronave.csv"

for ruta in (ruta_opensky, ruta_modelo_csv, ruta_aerolinea_csv):
    if not ruta.exists():
        raise FileNotFoundError(f"Falta '{ruta}'.")

# 2. Parámetros
FECHA_REFERENCIA = pd.Timestamp("2025-08-31")  # mes del dataset de OpenSky
PROB_MANTENIMIENTO = 0.02                       # SINTÉTICO: % de aeronaves en mantenimiento
SEMILLA = 42

# typecode OpenSky (designador ICAO) -> (fabricante, nombre) tal y como están en modelo_aeronave.csv
TYPECODE_A_MODELO = {
    "BCS3": ("Airbus", "A220-300"),
    "A319": ("Airbus", "A319-100"),
    "A320": ("Airbus", "A320-200"),
    "A20N": ("Airbus", "A320neo"),
    "A21N": ("Airbus", "A321neo"),
    "A332": ("Airbus", "A330-200"),
    "A333": ("Airbus", "A330-300"),
    "A359": ("Airbus", "A350-900"),
    "B738": ("Boeing", "737-800"),
    "B38M": ("Boeing", "737 MAX 8"),
    "B77W": ("Boeing", "777-300ER"),
    "B789": ("Boeing", "787-9"),
    "E190": ("Embraer", "E190"),
    "AT76": ("ATR", "72-600"),
    "CRJ9": ("Bombardier", "CRJ-900"),
}

# 3. Leer OpenSky (los valores van entre comillas simples)
df = pd.read_csv(
    ruta_opensky,
    quotechar="'",
    dtype=str,
    keep_default_na=False,
    usecols=["registration", "typecode", "operatorIcao", "operatorIata", "regUntil"],
    on_bad_lines="warn",
)
n_inicial = len(df)

# 4. Limpieza
df["registration"] = df["registration"].str.strip().str.upper()
df["typecode"] = df["typecode"].str.strip().str.upper()
df["operatorIcao"] = df["operatorIcao"].str.strip().str.upper()
df = df[(df["registration"] != "") & (df["registration"].str.len() <= 10)]
df = df.drop_duplicates(subset="registration")  # UNIQUE (matricula)

# 5. id_modelo: se asigna igual que lo hará AUTO_INCREMENT al cargar modelo_aeronave.csv (1..N en orden de fila)
df_modelo = pd.read_csv(ruta_modelo_csv)
if "id_modelo" not in df_modelo.columns:
    df_modelo["id_modelo"] = df_modelo.index + 1
mapa_modelo = {
    tc: int(df_modelo.loc[(df_modelo["fabricante"] == fab) & (df_modelo["nombre"] == nom), "id_modelo"].iloc[0])
    for tc, (fab, nom) in TYPECODE_A_MODELO.items()
    if ((df_modelo["fabricante"] == fab) & (df_modelo["nombre"] == nom)).any()
}

# 6. id_aerolinea: se cruza por código ICAO del operador.
#    (No se usa IATA: parte de tus códigos IATA son generados y podrían chocar con operadores reales.)
df_aero = pd.read_csv(ruta_aerolinea_csv, dtype={"codigo_icao": str})
df_aero = df_aero.dropna(subset=["codigo_icao"])
df_aero["codigo_icao"] = df_aero["codigo_icao"].str.strip().str.upper()
mapa_aerolinea = df_aero.drop_duplicates("codigo_icao").set_index("codigo_icao")["id_aerolinea"].to_dict()

df["id_aerolinea"] = df["operatorIcao"].map(mapa_aerolinea)
con_aerolinea = df.dropna(subset=["id_aerolinea"]).copy()

# Informe: tipos de avión que tienen aerolínea conocida pero NO están en modelo_aeronave
sin_modelo = con_aerolinea[~con_aerolinea["typecode"].isin(mapa_modelo)]
top_sin_modelo = sin_modelo["typecode"].replace("", "(vacío)").value_counts().head(20)

# 7. Quedarse solo con las aeronaves cuyo modelo y aerolínea existen en tus tablas
df = con_aerolinea[con_aerolinea["typecode"].isin(mapa_modelo)].copy()
df["id_modelo"] = df["typecode"].map(mapa_modelo).astype(int)
df["id_aerolinea"] = df["id_aerolinea"].astype(int)

# 8. Estado
# - retirada: matrícula caducada antes de la fecha de referencia (regUntil)
# - mantenimiento: SINTÉTICO, una pequeña fracción aleatoria de las operativas
# - operativa: el resto
reg_hasta = pd.to_datetime(df["regUntil"].str.strip(), errors="coerce")
df["estado"] = "operativa"
df.loc[reg_hasta.notna() & (reg_hasta < FECHA_REFERENCIA), "estado"] = "retirada"
rng = np.random.default_rng(SEMILLA)
operativas = df["estado"] == "operativa"
df.loc[operativas & (rng.random(len(df)) < PROB_MANTENIMIENTO), "estado"] = "mantenimiento"

# 9. Columnas finales
df = df.rename(columns={"registration": "matricula"})
df = df.sort_values(["id_aerolinea", "matricula"]).reset_index(drop=True)
df["id_aeronave"] = df.index + 1
df_final = df[["id_aeronave", "matricula", "id_modelo", "id_aerolinea", "estado"]]

# 10. Exportar
df_final.to_csv(ruta_aeronave_csv, index=False)

print(f"Filas leídas de OpenSky: {n_inicial}")
print(f"Aeronaves con aerolínea conocida (por ICAO): {len(con_aerolinea)}")
print(f"Aeronaves finales (aerolínea y modelo conocidos): {len(df_final)}")
print("\nPor estado:\n" + df_final["estado"].value_counts().to_string())
print("\nTipos de avión PERDIDOS por no estar en modelo_aeronave (amplíala con estos):")
print(top_sin_modelo.to_string())
print(f"\nArchivo guardado en {ruta_aeronave_csv}")
df_final.head(10)

Filas leídas de OpenSky: 616675
Aeronaves con aerolínea conocida (por ICAO): 33657
Aeronaves finales (aerolínea y modelo conocidos): 17143

Por estado:
estado
operativa        16784
mantenimiento      329
retirada            30

Tipos de avión PERDIDOS por no estar en modelo_aeronave (amplíala con estos):
typecode
A321       1664
B737        999
E75L        576
B763        517
CRJ2        469
B739        454
DH8D        423
B752        419
B772        418
TEX2        389
K35R        381
(vacío)     363
B744        351
B788        345
CRJ7        324
C130        313
E145        292
A388        249
B733        225
MI8         211

Archivo guardado en data\aeronave.csv


,id_aeronave,matricula,id_modelo,id_aerolinea,estado
0,1,F-HAAF,3,21,operativa
1,2,F-HAQD,3,21,operativa
2,3,F-HBAL,2,21,operativa
3,4,F-HBAO,3,21,operativa
4,5,F-HBAP,3,21,operativa
5,6,F-HBIB,3,21,operativa
6,7,F-HBIO,3,21,operativa
7,8,F-HBIS,3,21,operativa
8,9,F-HBIX,3,21,operativa
9,10,F-HBMI,2,21,operativa


In [14]:
from pathlib import Path

import numpy as np
import pandas as pd

# 1. Definir rutas
carpeta_raw = Path("raw_data")
carpeta_data = Path("data")
carpeta_data.mkdir(parents=True, exist_ok=True)

ruta_flights_csv = carpeta_raw / "flights.csv"   # <- cambia el nombre si el de Kaggle es otro
ruta_aeropuerto_csv = carpeta_data / "aeropuerto.csv"
ruta_aerolinea_csv = carpeta_data / "aerolinea.csv"
ruta_ruta_csv = carpeta_data / "ruta.csv"
ruta_aeronave_csv = carpeta_data / "aeronave.csv"
ruta_terminal_csv = carpeta_data / "terminal.csv"
ruta_puerta_csv = carpeta_data / "puerta_embarque.csv"
ruta_pista_csv = carpeta_data / "pista.csv"
ruta_vuelo_csv = carpeta_data / "vuelo.csv"

for ruta in (ruta_flights_csv, ruta_aeropuerto_csv, ruta_aerolinea_csv, ruta_ruta_csv, ruta_aeronave_csv,
             ruta_terminal_csv, ruta_puerta_csv, ruta_pista_csv):
    if not ruta.exists():
        raise FileNotFoundError(f"Falta '{ruta}'.")

TZ_ORIGEN = "America/New_York"   # el dataset solo tiene salidas de EWR, JFK y LGA
UMBRAL_RETRASO_MIN = 15          # criterio estándar: llegada con 15 min o más de retraso
OCUPACION_PUERTA_MIN = 45        # una puerta queda reservada 45 min antes de cada salida
SEMILLA = 42

# 2. Leer vuelos
df = pd.read_csv(ruta_flights_csv, dtype={"carrier": str, "tailnum": str, "origin": str, "dest": str})
n_inicial = len(df)
df = df.dropna(subset=["carrier", "flight", "origin", "dest", "sched_dep_time", "sched_arr_time", "tailnum"])
df["tailnum"] = df["tailnum"].str.strip().str.upper()

# 3. Tablas ya limpias
df_aero = pd.read_csv(ruta_aeropuerto_csv, usecols=["id_aeropuerto", "codigo_iata", "zona_horaria"])
df_aerolinea = pd.read_csv(ruta_aerolinea_csv, usecols=["id_aerolinea", "codigo_iata"]).dropna()
df_ruta = pd.read_csv(ruta_ruta_csv)           # columnas esperadas: id_ruta, origen, destino (ids de aeropuerto)
df_aeronave = pd.read_csv(ruta_aeronave_csv)   # id_aeronave, matricula, id_modelo, id_aerolinea, estado

# 4. id_ruta: IATA origen/destino -> id_aeropuerto -> ruta
print("Columnas de ruta.csv:", list(df_ruta.columns))
col_origen = next(c for c in df_ruta.columns if "orig" in c.lower())
col_destino = next(c for c in df_ruta.columns if "dest" in c.lower())
col_id_ruta = next(c for c in df_ruta.columns if c.lower().startswith("id_ruta"))
print(f"Usando: {col_id_ruta}, {col_origen}, {col_destino}")

iata_a_id = df_aero.drop_duplicates("codigo_iata").set_index("codigo_iata")["id_aeropuerto"]
df["id_origen"] = df["origin"].map(iata_a_id)
df["id_destino"] = df["dest"].map(iata_a_id)
df = df.dropna(subset=["id_origen", "id_destino"])
df["id_origen"] = df["id_origen"].astype(int)
df["id_destino"] = df["id_destino"].astype(int)

df_ruta = df_ruta.rename(columns={col_id_ruta: "id_ruta", col_origen: "ruta_origen", col_destino: "ruta_destino"})
df = df.merge(
    df_ruta[["id_ruta", "ruta_origen", "ruta_destino"]],
    left_on=["id_origen", "id_destino"], right_on=["ruta_origen", "ruta_destino"], how="inner",
)
n_tras_ruta = len(df)
if df.empty:
    raise ValueError("Ningún vuelo cruza con ruta.csv: revisa que origen/destino de ruta sean id_aeropuerto.")

# 5. id_aeronave: por matrícula, y solo si la aeronave pertenece hoy a la misma aerolínea que operó el vuelo
df = df.merge(df_aeronave[["id_aeronave", "matricula", "id_aerolinea"]],
              left_on="tailnum", right_on="matricula", how="inner")
n_tras_matricula = len(df)
if df.empty:
    raise ValueError("Ningún tailnum coincide con aeronave.matricula (aviones de 2013 frente a la base de 2025).")
id_aerolinea_carrier = df_aerolinea.drop_duplicates("codigo_iata").set_index("codigo_iata")["id_aerolinea"]
df = df[df["id_aerolinea"] == df["carrier"].map(id_aerolinea_carrier)]
n_tras_aerolinea = len(df)
print(f"Con ruta: {n_tras_ruta} | con matrícula: {n_tras_matricula} | misma aerolínea: {n_tras_aerolinea}")
if df.empty:
    raise ValueError("Ninguna aeronave pertenece hoy a la aerolínea que operó el vuelo.")

# 6. numero_vuelo = carrier + número (ej. UA1545)
df["numero_vuelo"] = df["carrier"].str.strip().str.upper() + df["flight"].astype(int).astype(str)
df = df[df["numero_vuelo"].str.len() <= 10]

# 7. Horas programadas. Se guardan TODAS en UTC para que sean comparables entre zonas horarias.
fecha = pd.to_datetime(dict(year=df["year"], month=df["month"], day=df["day"]))


def minutos(hhmm):
    hhmm = hhmm.astype(int)
    return pd.to_timedelta((hhmm // 100) * 60 + hhmm % 100, unit="m")


salida_local = fecha + minutos(df["sched_dep_time"])
df["salida_prevista"] = (
    salida_local.dt.tz_localize(TZ_ORIGEN, ambiguous="NaT", nonexistent="shift_forward").dt.tz_convert("UTC")
)

# sched_arr_time está en hora LOCAL del aeropuerto de destino
zona_destino = df["id_destino"].astype(int).map(df_aero.set_index("id_aeropuerto")["zona_horaria"])
llegada_local = fecha + minutos(df["sched_arr_time"])
llegada_utc = pd.Series(pd.NaT, index=df.index, dtype="datetime64[ns, UTC]")
for zona, idx in zona_destino.groupby(zona_destino).groups.items():
    llegada_utc.loc[idx] = (
        llegada_local.loc[idx].dt.tz_localize(zona, ambiguous="NaT", nonexistent="shift_forward").dt.tz_convert("UTC")
    )
df["llegada_prevista"] = llegada_utc
df = df.dropna(subset=["salida_prevista", "llegada_prevista"]).copy()

# Si la llegada cae antes que la salida (vuelo que cruza medianoche), suma días
for _ in range(2):
    mal = df["llegada_prevista"] <= df["salida_prevista"]
    df.loc[mal, "llegada_prevista"] += pd.Timedelta(days=1)
df = df[df["llegada_prevista"] > df["salida_prevista"]]

# 8. Horas reales = prevista + retraso. Sin dep_time o sin arr_time: cancelado (o desviado) y sin horas reales
df["salida_real"] = df["salida_prevista"] + pd.to_timedelta(df["dep_delay"], unit="m")
df["llegada_real"] = df["llegada_prevista"] + pd.to_timedelta(df["arr_delay"], unit="m")
sin_datos = df["dep_time"].isna() | df["arr_time"].isna() | df["arr_delay"].isna()
df.loc[sin_datos, ["salida_real", "llegada_real"]] = pd.NaT

# 9. Estado
df["estado"] = np.where(
    sin_datos, "cancelado",
    np.where(df["arr_delay"] >= UMBRAL_RETRASO_MIN, "retrasado", "aterrizado"),
)

# 11. Quitar duplicados de la UNIQUE (numero_vuelo, salida_prevista) y crear id_vuelo
df = df.drop_duplicates(subset=["numero_vuelo", "salida_prevista"])
df = df.sort_values("salida_prevista").reset_index(drop=True)
df["id_vuelo"] = df.index + 1

# 12. id_puerta e id_pista (de SALIDA) a partir de puerta_embarque.csv y pista.csv.
#     Solo hay puerta/pista si el aeropuerto de origen existe en esos CSV; si no, queda NULL.
#     La elección entre las disponibles del aeropuerto es ALEATORIA (sintética), no el dato real.
rng = np.random.default_rng(SEMILLA)
df_terminal = pd.read_csv(ruta_terminal_csv, usecols=["id_terminal", "id_aeropuerto"])
df_puerta = pd.read_csv(ruta_puerta_csv, usecols=["id_puerta", "id_terminal"]).merge(df_terminal, on="id_terminal")
df_pista = pd.read_csv(ruta_pista_csv, usecols=["id_pista", "id_aeropuerto", "longitud_m"])

puertas_por_aero = {a: g["id_puerta"].to_numpy() for a, g in df_puerta.groupby("id_aeropuerto")}
pistas_por_aero = {
    a: (g["id_pista"].to_numpy(), (g["longitud_m"] / g["longitud_m"].sum()).to_numpy())  # más peso a pistas largas
    for a, g in df_pista.groupby("id_aeropuerto")
}

salida_min = (df["salida_prevista"].astype("int64") // 60_000_000_000).to_numpy()  # minutos desde epoch (UTC)
origen_ids = df["id_origen"].to_numpy()
id_puerta = np.full(len(df), np.nan)
id_pista = np.full(len(df), np.nan)
libre_desde = {}  # id_puerta -> minuto a partir del cual queda libre

for i in range(len(df)):  # df ya está ordenado por salida_prevista
    ao = origen_ids[i]
    if ao in puertas_por_aero:
        libres = [g for g in puertas_por_aero[ao] if libre_desde.get(g, -1) <= salida_min[i] - OCUPACION_PUERTA_MIN]
        if libres:
            g = rng.choice(libres)
            id_puerta[i] = g
            libre_desde[g] = salida_min[i]
    if ao in pistas_por_aero:
        ids, pesos = pistas_por_aero[ao]
        id_pista[i] = rng.choice(ids, p=pesos)

df["id_puerta"] = pd.Series(id_puerta, index=df.index).astype("Int64")
df["id_pista"] = pd.Series(id_pista, index=df.index).astype("Int64")



def fmt(col):
    return col.dt.tz_convert("UTC").dt.strftime("%Y-%m-%d %H:%M:%S")


for c in ("salida_prevista", "llegada_prevista", "salida_real", "llegada_real"):
    df[c] = fmt(df[c])

columnas_finales = [
    "id_vuelo", "numero_vuelo", "id_ruta", "id_aeronave", "id_puerta", "id_pista",
    "salida_prevista", "llegada_prevista", "salida_real", "llegada_real", "estado",
]
df_final = df[columnas_finales]

# 12. Exportar (los NULL salen como celda vacía; en LOAD DATA usa NULLIF o SET para convertirlos)
df_final.to_csv(ruta_vuelo_csv, index=False)

print(f"Vuelos leídos: {n_inicial}")
print(f"Con ruta en tu tabla ruta: {n_tras_ruta}")
print(f"Con matrícula en aeronave: {n_tras_matricula}")
print(f"Con la misma aerolínea que hoy: {n_tras_aerolinea}")
print(f"Vuelos finales: {len(df_final)}")
print(f"Con id_puerta: {df_final['id_puerta'].notna().sum()} | con id_pista: {df_final['id_pista'].notna().sum()}")
print(df_final["estado"].value_counts().to_string())
print(f"Archivo guardado en {ruta_vuelo_csv}")
df_final.head(10)

Columnas de ruta.csv: ['id_ruta', 'id_aeropuerto_origen', 'id_aeropuerto_destino', 'distancia_km']
Usando: id_ruta, id_aeropuerto_origen, id_aeropuerto_destino
Con ruta: 331152 | con matrícula: 138561 | misma aerolínea: 114892
Vuelos leídos: 336776
Con ruta en tu tabla ruta: 331152
Con matrícula en aeronave: 138561
Con la misma aerolínea que hoy: 114892
Vuelos finales: 114892
Con id_puerta: 1308 | con id_pista: 114892
estado
aterrizado    87068
retrasado     26819
cancelado      1005
Archivo guardado en data\vuelo.csv


,id_vuelo,numero_vuelo,id_ruta,id_aeronave,id_puerta,id_pista,salida_prevista,llegada_prevista,salida_real,llegada_real,estado
0,1,UA1545,33837,14605,134,240934,2013-01-01 10:15:00,2013-01-01 14:19:00,2013-01-01 10:17:00,2013-01-01 14:30:00,aterrizado
1,2,UA1714,15850,14702,<NA>,244204,2013-01-01 10:29:00,2013-01-01 14:30:00,2013-01-01 10:33:00,2013-01-01 14:50:00,retrasado
2,3,B6725,11464,8701,<NA>,242385,2013-01-01 10:45:00,2013-01-01 14:22:00,2013-01-01 10:44:00,2013-01-01 14:04:00,aterrizado
3,4,B61806,5119,8684,<NA>,242384,2013-01-01 10:59:00,2013-01-01 12:06:00,2013-01-01 10:59:00,2013-01-01 12:02:00,aterrizado
4,5,B6343,11414,8664,209,240934,2013-01-01 11:00:00,2013-01-01 13:50:00,2013-01-01 11:01:00,2013-01-01 13:44:00,aterrizado
5,6,B6371,11511,8633,<NA>,244204,2013-01-01 11:00:00,2013-01-01 13:58:00,2013-01-01 11:00:00,2013-01-01 13:51:00,aterrizado
6,7,UA1187,33841,14982,202,240933,2013-01-01 11:00:00,2013-01-01 17:02:00,2013-01-01 10:59:00,2013-01-01 16:54:00,aterrizado
7,8,B6125,11472,8645,<NA>,242384,2013-01-01 11:00:00,2013-01-01 14:01:00,NaN,NaN,cancelado
8,9,B649,11485,8698,<NA>,242384,2013-01-01 11:00:00,2013-01-01 13:51:00,2013-01-01 10:58:00,2013-01-01 13:49:00,aterrizado
9,10,B679,5147,8631,<NA>,242385,2013-01-01 11:00:00,2013-01-01 13:46:00,2013-01-01 10:57:00,2013-01-01 13:38:00,aterrizado


In [15]:
from pathlib import Path

import pandas as pd

# 1. Definir rutas
carpeta_data = Path("data")
ruta_vuelo_csv = carpeta_data / "vuelo.csv"
ruta_evento_csv = carpeta_data / "evento_vuelo.csv"

if not ruta_vuelo_csv.exists():
    raise FileNotFoundError("Falta 'vuelo.csv' en la carpeta data/. Genéralo antes de procesar los eventos.")

# 2. Reglas (minutos respecto a la salida). Los eventos de preparación son DERIVADOS, no datos reales.
MIN_CHECKIN_ABIERTO = 120   # el check-in abre 2 h antes de la salida prevista
MIN_EMBARQUE = 40           # el embarque empieza 40 min antes de la salida real
MIN_CIERRE_PUERTA = 15      # la puerta se cierra 15 min antes de la salida real
MIN_CANCELACION = 60        # la cancelación se registra 60 min antes de la salida prevista
UMBRAL_RETRASO_MIN = 15     # retraso de salida a partir del cual se registra un evento de retraso

# 3. Leer vuelos
df = pd.read_csv(
    ruta_vuelo_csv,
    usecols=["id_vuelo", "salida_prevista", "salida_real", "llegada_real", "estado"],
    parse_dates=["salida_prevista", "salida_real", "llegada_real"],
)
n_vuelos = len(df)
min_ = lambda n: pd.to_timedelta(n, unit="m")

cancelado = df["estado"] == "cancelado"
normal = ~cancelado

bloques = []


def anadir(mascara, tipo, fecha_hora):
    b = pd.DataFrame({"id_vuelo": df.loc[mascara, "id_vuelo"], "tipo_evento": tipo, "fecha_hora": fecha_hora[mascara]})
    bloques.append(b.dropna(subset=["fecha_hora"]))


# 4. Vuelos operados
anadir(normal, "checkin_abierto", df["salida_prevista"] - min_(MIN_CHECKIN_ABIERTO))
anadir(normal, "embarque_iniciado", df["salida_real"] - min_(MIN_EMBARQUE))
anadir(normal, "puerta_cerrada", df["salida_real"] - min_(MIN_CIERRE_PUERTA))
anadir(normal, "salida_puerta", df["salida_real"])          # real: salida_real
anadir(normal, "llegada_puerta", df["llegada_real"])        # real: llegada_real

# 5. Retrasos de salida (solo si la salida real supera a la prevista en el umbral)
retraso_min = (df["salida_real"] - df["salida_prevista"]).dt.total_seconds() / 60
anadir(normal & (retraso_min >= UMBRAL_RETRASO_MIN), "retraso_notificado", df["salida_prevista"])

# 6. Vuelos cancelados: solo el evento de cancelación
anadir(cancelado, "vuelo_cancelado", df["salida_prevista"] - min_(MIN_CANCELACION))

# 7. Unir, ordenar cronológicamente y crear id_evento (clave primaria junto con fecha_hora)
ev = pd.concat(bloques, ignore_index=True)
ev = ev.sort_values(["fecha_hora", "id_vuelo", "tipo_evento"]).reset_index(drop=True)
ev["id_evento"] = ev.index + 1
ev["fecha_hora"] = ev["fecha_hora"].dt.strftime("%Y-%m-%d %H:%M:%S")
ev = ev[ev["tipo_evento"].str.len() <= 30]

df_final = ev[["id_evento", "id_vuelo", "tipo_evento", "fecha_hora"]]

# 8. Exportar
df_final.to_csv(ruta_evento_csv, index=False)

print(f"Vuelos procesados: {n_vuelos}")
print(f"Eventos generados: {len(df_final)}")
print(df_final["tipo_evento"].value_counts().to_string())
print(f"Archivo guardado en {ruta_evento_csv}")
df_final.head(10)

Vuelos procesados: 114892
Eventos generados: 594957
tipo_evento
checkin_abierto       113887
embarque_iniciado     113887
puerta_cerrada        113887
salida_puerta         113887
llegada_puerta        113887
retraso_notificado     24517
vuelo_cancelado         1005
Archivo guardado en data\evento_vuelo.csv


,id_evento,id_vuelo,tipo_evento,fecha_hora
0,1,1,checkin_abierto,2013-01-01 08:15:00
1,2,2,checkin_abierto,2013-01-01 08:29:00
2,3,3,checkin_abierto,2013-01-01 08:45:00
3,4,4,checkin_abierto,2013-01-01 08:59:00
4,5,5,checkin_abierto,2013-01-01 09:00:00
5,6,6,checkin_abierto,2013-01-01 09:00:00
6,7,7,checkin_abierto,2013-01-01 09:00:00
7,8,9,checkin_abierto,2013-01-01 09:00:00
8,9,10,checkin_abierto,2013-01-01 09:00:00
9,10,11,checkin_abierto,2013-01-01 09:00:00


In [ ]:
from pathlib import Path

import pandas as pd
from faker import Faker

# 1. Definir rutas
carpeta_data = Path("data")
ruta_vuelo_csv = carpeta_data / "vuelo.csv"
ruta_aeronave_csv = carpeta_data / "aeronave.csv"
ruta_empleado_csv = carpeta_data / "empleado.csv"

for ruta in (ruta_vuelo_csv, ruta_aeronave_csv):
    if not ruta.exists():
        raise FileNotFoundError(f"Falta '{ruta}'. Genéralo antes de crear los empleados.")

# 2. Parámetros (SINTÉTICOS: ajusta los empleados por aeronave a tu gusto)
EMPLEADOS_POR_AERONAVE = {"piloto": 4, "copiloto": 4, "tcp": 12, "mecanico": 3}
MINIMO_POR_ROL = 4          # mínimo de empleados de cada rol por aerolínea
SEMILLA = 42
PREFIJO_LICENCIA = {"piloto": "ATPL", "copiloto": "CPL", "tcp": "CCA", "mecanico": "B1"}

Faker.seed(SEMILLA)
fake = Faker("es_ES")

# 3. Aerolíneas con vuelos y número de aeronaves que usan (para dimensionar la plantilla)
vuelos = pd.read_csv(ruta_vuelo_csv, usecols=["id_aeronave"])
aeronaves = pd.read_csv(ruta_aeronave_csv, usecols=["id_aeronave", "id_aerolinea"])
usadas = aeronaves[aeronaves["id_aeronave"].isin(vuelos["id_aeronave"].unique())]
flota = usadas.groupby("id_aerolinea")["id_aeronave"].nunique()

# 4. Generar empleados
filas = []
licencias_usadas = set()


def nueva_licencia(rol):
    while True:
        lic = f"{PREFIJO_LICENCIA[rol]}-{fake.random_int(10_000_000, 99_999_999)}"
        if lic not in licencias_usadas:
            licencias_usadas.add(lic)
            return lic


for id_aerolinea, n_aeronaves in flota.items():
    for rol, por_aeronave in EMPLEADOS_POR_AERONAVE.items():
        cantidad = max(MINIMO_POR_ROL, n_aeronaves * por_aeronave)
        for _ in range(cantidad):
            filas.append({
                "nombre": fake.first_name()[:50],
                "apellidos": f"{fake.last_name()} {fake.last_name()}"[:100],
                "rol": rol,
                "licencia": nueva_licencia(rol),
                "id_aerolinea": int(id_aerolinea),
            })

df = pd.DataFrame(filas)

# 5. Ordenar y crear id_empleado
orden_rol = {"piloto": 0, "copiloto": 1, "tcp": 2, "mecanico": 3}
df = df.assign(_o=df["rol"].map(orden_rol)).sort_values(["id_aerolinea", "_o"]).drop(columns="_o")
df = df.reset_index(drop=True)
df["id_empleado"] = df.index + 1

# 6. Columnas finales, iguales al CREATE TABLE
df_final = df[["id_empleado", "nombre", "apellidos", "rol", "licencia", "id_aerolinea"]]

# 7. Exportar
df_final.to_csv(ruta_empleado_csv, index=False)

print(f"Aerolíneas con vuelos: {len(flota)} | aeronaves usadas: {int(flota.sum())}")
print(f"Empleados generados: {len(df_final)}")
print(df_final["rol"].value_counts().to_string())
print(f"Archivo guardado en {ruta_empleado_csv}")
df_final.head(10)

   ---------------------------------------- 0.0/2.1 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.1 MB 682.7 kB/s eta 0:00:04
   ---- ----------------------------------- 0.3/2.1 MB 4.0 MB/s eta 0:00:01
   ---------------------------------------  2.1/2.1 MB 18.9 MB/s eta 0:00:01
   ---------------------------------------- 2.1/2.1 MB 16.5 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


Aerolíneas con vuelos: 10 | aeronaves usadas: 855
Empleados generados: 19665
rol
tcp         10260
piloto       3420
copiloto     3420
mecanico     2565
Archivo guardado en data\empleado.csv


,id_empleado,nombre,apellidos,rol,licencia,id_aerolinea
0,1,Feliciana,Cantón Amaya,piloto,ATPL-46913810,452
1,2,Jose Ignacio,Godoy Chaparro,piloto,ATPL-23756669,452
2,3,Inmaculada,Bonet Roca,piloto,ATPL-14265799,452
3,4,Amor,Bueno Gil,piloto,ATPL-41227216,452
4,5,Anselma,Amor Flor,piloto,ATPL-97226012,452
5,6,Judith,Rivas Giménez,piloto,ATPL-70291817,452
6,7,Dolores,Lobo Agudo,piloto,ATPL-31429110,452
7,8,José,Rocamora Nieto,piloto,ATPL-47295260,452
8,9,Fabricio,Garrido Navarro,piloto,ATPL-23718431,452
9,10,Cirino,Pinto Caballero,piloto,ATPL-58181396,452


In [18]:
from pathlib import Path
import heapq
import math

import numpy as np
import pandas as pd

# 1. Definir rutas
carpeta_data = Path("data")
ruta_vuelo_csv = carpeta_data / "vuelo.csv"
ruta_aeronave_csv = carpeta_data / "aeronave.csv"
ruta_modelo_csv = carpeta_data / "modelo_aeronave.csv"
ruta_empleado_csv = carpeta_data / "empleado.csv"
ruta_tripulacion_csv = carpeta_data / "tripulacion_vuelo.csv"

for ruta in (ruta_vuelo_csv, ruta_aeronave_csv, ruta_modelo_csv, ruta_empleado_csv):
    if not ruta.exists():
        raise FileNotFoundError(f"Falta '{ruta}'. Genéralo antes de asignar tripulaciones.")

# 2. Parámetros (SINTÉTICOS salvo la regla de dotación de TCP)
PLAZAS_POR_TCP = 50          # 1 TCP por cada 50 plazas (regla de dotación mínima; contrástala antes de citarla)
MIN_ANTES_SALIDA = 60        # la tripulación se presenta 60 min antes de la salida prevista
MIN_DESCANSO_TRAS_VUELO = 45 # tiempo mínimo entre el fin de un vuelo y el siguiente del mismo empleado
SEMILLA = 42

rng = np.random.default_rng(SEMILLA)

# 3. Leer datos
vuelo = pd.read_csv(
    ruta_vuelo_csv,
    usecols=["id_vuelo", "id_aeronave", "salida_prevista", "llegada_prevista", "llegada_real", "estado"],
    parse_dates=["salida_prevista", "llegada_prevista", "llegada_real"],
)
aeronave = pd.read_csv(ruta_aeronave_csv, usecols=["id_aeronave", "id_modelo", "id_aerolinea"])
modelo = pd.read_csv(ruta_modelo_csv)
if "id_modelo" not in modelo.columns:
    modelo["id_modelo"] = modelo.index + 1   # igual que AUTO_INCREMENT al cargar el CSV entero
empleado = pd.read_csv(ruta_empleado_csv, usecols=["id_empleado", "rol", "id_aerolinea"])

df = (
    vuelo.merge(aeronave, on="id_aeronave", how="left")
    .merge(modelo[["id_modelo", "capacidad_total"]], on="id_modelo", how="left")
)
df = df[df["estado"] != "cancelado"].sort_values("salida_prevista").reset_index(drop=True)  # un vuelo cancelado no se tripula


def a_minutos(serie):
    return (serie.astype("int64") // 60_000_000_000)


salida = a_minutos(df["salida_prevista"]).to_numpy()
fin_vuelo = a_minutos(df["llegada_real"].fillna(df["llegada_prevista"])).to_numpy()

# 4. Un montón (heap) por (aerolínea, rol): el empleado libre desde hace más tiempo sale primero.
#    Elemento: (minuto en que queda libre, desempate aleatorio, id_empleado)
heaps = {}
for (aerolinea, rol), g in empleado.groupby(["id_aerolinea", "rol"]):
    ids = g["id_empleado"].to_numpy()
    items = [(-10**12, float(rng.random()), int(i)) for i in ids]
    heapq.heapify(items)
    heaps[(int(aerolinea), rol)] = items


def tomar(aerolinea, rol, cantidad, inicio):
    """Saca hasta 'cantidad' empleados libres a la hora 'inicio'."""
    heap = heaps.get((aerolinea, rol))
    elegidos = []
    while heap and len(elegidos) < cantidad and heap[0][0] <= inicio:
        elegidos.append(heapq.heappop(heap))
    return elegidos


def devolver(aerolinea, rol, elegidos, libre_desde):
    heap = heaps[(aerolinea, rol)]
    for _, desempate, emp in elegidos:
        heapq.heappush(heap, (libre_desde, desempate, emp))


# 5. Asignar tripulación vuelo a vuelo, en orden cronológico
filas = []
faltan = {"comandante": 0, "copiloto": 0, "tcp": 0}
sin_modelo = 0

for i, r in enumerate(df.itertuples(index=False)):
    if pd.isna(r.id_aerolinea) or pd.isna(r.capacidad_total):
        sin_modelo += 1
        continue
    aerolinea = int(r.id_aerolinea)
    inicio = salida[i] - MIN_ANTES_SALIDA
    libre_desde = fin_vuelo[i] + MIN_DESCANSO_TRAS_VUELO
    n_tcp = max(2, math.ceil(r.capacidad_total / PLAZAS_POR_TCP))  # incluye al jefe de cabina

    for rol, cantidad in (("piloto", 1), ("copiloto", 1), ("tcp", n_tcp)):
        elegidos = tomar(aerolinea, rol, cantidad, inicio)
        for k, (_, _, emp) in enumerate(elegidos):
            if rol == "piloto":
                funcion = "comandante"
            elif rol == "copiloto":
                funcion = "copiloto"
            else:
                funcion = "jefe_cabina" if k == 0 else "tcp"
            filas.append((int(r.id_vuelo), emp, funcion))
        if len(elegidos) < cantidad:
            faltan["comandante" if rol == "piloto" else rol] += cantidad - len(elegidos)
        if elegidos:
            devolver(aerolinea, rol, elegidos, libre_desde)

# 6. Columnas finales, iguales al CREATE TABLE
df_final = pd.DataFrame(filas, columns=["id_vuelo", "id_empleado", "funcion"])
df_final = df_final.drop_duplicates(subset=["id_vuelo", "id_empleado"]).sort_values(["id_vuelo", "id_empleado"])

# 7. Exportar
df_final.to_csv(ruta_tripulacion_csv, index=False)

print(f"Vuelos tripulados (sin cancelados): {len(df)} | sin aerolínea o modelo: {sin_modelo}")
print(f"Filas generadas: {len(df_final)}")
print(df_final["funcion"].value_counts().to_string())
print(f"Plazas SIN cubrir por falta de empleados libres: {faltan}")
if any(faltan.values()):
    print("-> Sube EMPLEADOS_POR_AERONAVE en generar_empleado.py y vuelve a generar empleados y tripulación.")
print(f"Archivo guardado en {ruta_tripulacion_csv}")
df_final.head(10)

Vuelos tripulados (sin cancelados): 113887 | sin aerolínea o modelo: 0
Filas generadas: 80432
funcion
tcp            34738
comandante     16056
copiloto       16056
jefe_cabina    13582
Plazas SIN cubrir por falta de empleados libres: {'comandante': 97831, 'copiloto': 97831, 'tcp': 353051}
-> Sube EMPLEADOS_POR_AERONAVE en generar_empleado.py y vuelve a generar empleados y tripulación.
Archivo guardado en data\tripulacion_vuelo.csv


,id_vuelo,id_empleado,funcion
0,1,12508,comandante
1,1,13579,copiloto
5,1,14403,tcp
4,1,15090,tcp
2,1,15394,jefe_cabina
3,1,17220,tcp
6,2,12353,comandante
7,2,14290,copiloto
8,2,14894,jefe_cabina
11,2,16103,tcp


In [19]:
from pathlib import Path
import re
import unicodedata

import numpy as np
import pandas as pd
from faker import Faker

# 1. Definir rutas
carpeta_data = Path("data")
ruta_pais_csv = carpeta_data / "pais.csv"
ruta_vuelo_csv = carpeta_data / "vuelo.csv"          # opcional: solo para fijar la fecha de referencia
ruta_pasajero_csv = carpeta_data / "pasajero.csv"

if not ruta_pais_csv.exists():
    raise FileNotFoundError("Falta 'pais.csv' en la carpeta data/.")

# 2. Parámetros (TODO SINTÉTICO)
NUM_PASAJEROS = 100_000
SEMILLA = 42
PROB_SIN_EMAIL = 0.08
PERFIL = "espana"   # "espana": nombres es_ES y mezcla de nacionalidades centrada en España (lo que pediste)
                    # "nueva_york": mezcla centrada en EE. UU. y nombres según el país (coherente con vuelos de NYC)

PESOS = {
    "espana": {"ES": 55, "GB": 8, "DE": 7, "FR": 6, "IT": 5, "PT": 4, "NL": 3, "US": 4, "IE": 2,
               "BE": 1.5, "CH": 1.5, "SE": 1, "NO": 1, "DK": 1, "PL": 1},
    "nueva_york": {"US": 60, "GB": 5, "CA": 4, "DE": 4, "FR": 4, "IT": 4, "ES": 4, "MX": 3, "BR": 2,
                   "IN": 2, "CN": 2, "JP": 2, "IE": 2, "PT": 1, "NL": 1},
}[PERFIL]
LOCALES_POR_PAIS = {"US": "en_US", "GB": "en_GB", "CA": "en_CA", "IE": "en_IE", "DE": "de_DE", "FR": "fr_FR",
                    "IT": "it_IT", "PT": "pt_PT", "NL": "nl_NL", "BR": "pt_BR", "MX": "es_MX", "IN": "en_IN",
                    "CN": "zh_CN", "JP": "ja_JP"}
USAR_LOCALE_POR_PAIS = PERFIL == "nueva_york"   # si es False, todos los nombres salen de Faker es_ES
DOMINIOS = ["gmail.com", "hotmail.com", "outlook.com", "yahoo.es", "icloud.com", "protonmail.com"]
PESOS_DOMINIOS = [0.5, 0.2, 0.12, 0.1, 0.05, 0.03]

rng = np.random.default_rng(SEMILLA)
Faker.seed(SEMILLA)
fakers = {}


def faker_para(iso):
    loc = LOCALES_POR_PAIS.get(iso, "es_ES") if USAR_LOCALE_POR_PAIS else "es_ES"
    if loc not in fakers:
        fakers[loc] = Faker(loc)
    return fakers[loc]


# 3. Países disponibles (id_pais por código ISO) y pesos normalizados
df_pais = pd.read_csv(ruta_pais_csv, dtype={"codigo_iso": str})
iso_a_id = df_pais.dropna(subset=["codigo_iso"]).drop_duplicates("codigo_iso").set_index("codigo_iso")["id_pais"].to_dict()
pesos = {k: v for k, v in PESOS.items() if k in iso_a_id}
faltan_pais = sorted(set(PESOS) - set(pesos))
isos = list(pesos)
prob = np.array([pesos[k] for k in isos], dtype=float)
prob /= prob.sum()

# 4. Fecha de referencia para calcular las edades (la de los vuelos, si existen)
if ruta_vuelo_csv.exists():
    ref = pd.to_datetime(pd.read_csv(ruta_vuelo_csv, usecols=["salida_prevista"])["salida_prevista"]).min().normalize()
else:
    ref = pd.Timestamp("2013-01-01")

LETRAS_DNI = "TRWAGMYFPDXBNJZSQVHLCKE"
LETRAS = np.array(list("ABCDEFGHJKLMNPQRSTUVWXYZ"))


def documento(iso, usados):
    while True:
        if iso == "ES":
            n = int(rng.integers(10_000_000, 99_999_999))
            doc = f"{n}{LETRAS_DNI[n % 23]}"              # DNI con letra de control válida (número inventado)
        else:
            doc = "".join(rng.choice(LETRAS, 2)) + f"{int(rng.integers(0, 10_000_000)):07d}"  # pasaporte inventado
        if doc not in usados:
            usados.add(doc)
            return doc


def limpiar(txt):
    txt = unicodedata.normalize("NFKD", txt).encode("ascii", "ignore").decode()
    return re.sub(r"[^a-z0-9]", "", txt.lower())


# 5. Generar pasajeros
filas = []
usados = set()
paises = rng.choice(isos, size=NUM_PASAJEROS, p=prob)
edades = np.clip(rng.normal(40, 17, NUM_PASAJEROS), 2, 90)
dias_extra = rng.integers(0, 365, NUM_PASAJEROS)

for i in range(NUM_PASAJEROS):
    iso = paises[i]
    f = faker_para(iso)
    nombre = f.first_name()[:50]
    if iso in ("ES", "MX", "BR", "PT") or not USAR_LOCALE_POR_PAIS:
        apellidos = f"{f.last_name()} {f.last_name()}"
    else:
        apellidos = f.last_name()
    nacimiento = (ref - pd.DateOffset(years=int(edades[i]))) - pd.Timedelta(days=int(dias_extra[i]))

    email = None
    if rng.random() >= PROB_SIN_EMAIL:
        local = f"{limpiar(nombre.split()[0])}.{limpiar(apellidos.split()[0])}"
        if rng.random() < 0.6:
            local += str(int(rng.integers(1, 9999)))
        email = f"{local}@{rng.choice(DOMINIOS, p=PESOS_DOMINIOS)}"[:100]

    filas.append({
        "nombre": nombre,
        "apellidos": apellidos[:100],
        "documento": documento(iso, usados),
        "fecha_nacimiento": nacimiento.strftime("%Y-%m-%d"),
        "id_pais": int(iso_a_id[iso]),
        "email": email,
    })

df = pd.DataFrame(filas)
df["id_pasajero"] = df.index + 1

# 6. Columnas finales, iguales al CREATE TABLE
df_final = df[["id_pasajero", "nombre", "apellidos", "documento", "fecha_nacimiento", "id_pais", "email"]]

# 7. Exportar
df_final.to_csv(ruta_pasajero_csv, index=False)

print(f"Perfil: {PERFIL} | pasajeros generados: {len(df_final)} | fecha de referencia de edades: {ref.date()}")
if faltan_pais:
    print(f"Países sin fila en pais.csv (ignorados): {faltan_pais}")
print(df.assign(iso=paises)["iso"].value_counts().head(8).to_string())
print(f"Archivo guardado en {ruta_pasajero_csv}")
df_final.head(10)

Perfil: espana | pasajeros generados: 100000 | fecha de referencia de edades: 2013-01-01
iso
ES    54281
GB     7982
DE     7049
FR     5948
IT     5032
US     3928
PT     3898
NL     2934
Archivo guardado en data\pasajero.csv


,id_pasajero,nombre,apellidos,documento,fecha_nacimiento,id_pais,email
0,1,Feliciana,Cantón Amaya,RB8630072,1954-06-14,108,feliciana.canton3896@outlook.com
1,2,Manuela,Llopis Hierro,26245491F,1995-02-19,197,manuela.llopis1819@outlook.com
2,3,Jacinto,Chaparro Calatayud,MH7500806,1963-07-05,159,jacinto.chaparro5799@gmail.com
3,4,Inmaculada,Bonet Roca,SX6073139,1972-02-13,78,inmaculada.bonet@hotmail.com
4,5,Amílcar,Andrés Bueno,50437065C,1932-11-25,197,amilcar.andres@gmail.com
5,6,Isaías,Guerrero Vazquez,DB9401582,1960-07-11,160,isaias.guerrero@gmail.com
6,7,Eligia,Amor Flor,WT3613506,1972-07-29,108,NaN
7,8,Lilia,Rivas Giménez,SD3622443,1955-06-24,108,lilia.rivas9@gmail.com
8,9,Vidal,Lobo Agudo,82037928X,1963-01-16,197,vidal.lobo5762@gmail.com
9,10,Martina,Cuervo Rocamora,85476519D,1960-10-20,197,martina.cuervo1795@gmail.com


In [20]:
from pathlib import Path
import math

import numpy as np
import pandas as pd

# 1. Definir rutas
carpeta_data = Path("data")
nombres = ["vuelo", "aeronave", "modelo_aeronave", "ruta", "clase_cabina", "pasajero"]
rutas_in = {n: carpeta_data / f"{n}.csv" for n in nombres}
for n, r in rutas_in.items():
    if not r.exists():
        raise FileNotFoundError(f"Falta '{r}'. Genéralo antes de ejecutar este script.")

# 2. Parámetros (TODO LO QUE SE GENERA AQUÍ ES SINTÉTICO, calibrado con los vuelos reales del dataset)
NUM_VUELOS_MUESTRA = 5000   # vuelos con billetes/equipaje. None = todos (decenas de millones de filas)
SEMILLA = 42
OCUPACION_MEDIA = 0.82      # factor de ocupación medio por vuelo
PROB_RESERVA_CANCELADA = 0.07
PROB_NO_SHOW = 0.04
PROB_PERDIDA_MALETA = 0.006
PROB_INCIDENCIA_ALEATORIA = 0.004
TARIFA_BASE = 40.0          # precio = (base + tarifa_km * km) * multiplicador de clase * ruido
TARIFA_KM = 0.11
EPOCH = pd.Timestamp("1970-01-01")

rng = np.random.default_rng(SEMILLA)


def minutos(serie):
    """Minutos desde 1970, independiente de la resolución (ns/us) que use pandas."""
    return ((serie - EPOCH) // pd.Timedelta(minutes=1)).astype("int64")


def texto(valores):
    return pd.to_datetime(pd.Series(valores, dtype="float64"), unit="m").dt.strftime("%Y-%m-%d %H:%M:%S")


# 3. Leer datos
vuelo = pd.read_csv(
    rutas_in["vuelo"],
    usecols=["id_vuelo", "id_ruta", "id_aeronave", "salida_prevista", "llegada_prevista",
             "salida_real", "llegada_real", "estado"],
    parse_dates=["salida_prevista", "llegada_prevista", "salida_real", "llegada_real"],
)
aeronave = pd.read_csv(rutas_in["aeronave"])
modelo = pd.read_csv(rutas_in["modelo_aeronave"])
if "id_modelo" not in modelo.columns:
    modelo["id_modelo"] = modelo.index + 1
ruta = pd.read_csv(rutas_in["ruta"])
col_dist = next(c for c in ruta.columns if "dist" in c.lower())
col_id_ruta = next(c for c in ruta.columns if c.lower().startswith("id_ruta"))
clase = pd.read_csv(rutas_in["clase_cabina"])
if "id_clase" not in clase.columns:
    clase["id_clase"] = clase.index + 1
pasajero = pd.read_csv(rutas_in["pasajero"], usecols=["id_pasajero", "fecha_nacimiento"], parse_dates=["fecha_nacimiento"])

df = (
    vuelo.merge(aeronave[["id_aeronave", "id_modelo"]], on="id_aeronave", how="left")
    .merge(modelo[["id_modelo", "capacidad_total"]], on="id_modelo", how="left")
    .merge(ruta[[col_id_ruta, col_dist]].rename(columns={col_id_ruta: "id_ruta", col_dist: "dist_km"}), on="id_ruta", how="left")
)
df["sal"] = minutos(df["salida_prevista"])
df["lleg_prev"] = minutos(df["llegada_prevista"])
df["sal_real"] = minutos(df["salida_real"].fillna(df["salida_prevista"]))
df["lleg"] = minutos(df["llegada_real"].fillna(df["llegada_prevista"]))
t0, t1 = int(df["sal"].min()), int(df["lleg"].max())

# Clases de cabina: id por palabra clave del nombre y multiplicador de precio por categoría
def buscar_clase(claves, excluir=()):
    for _, r in clase.iterrows():
        nom = str(r["nombre"]).lower()
        if any(k in nom for k in claves) and not any(x in nom for x in excluir):
            return int(r["id_clase"])
    return None


MULT = {"eco": 1.0, "premium": 1.6, "business": 3.2, "primera": 5.0}
ids_clase = {
    "eco": buscar_clase(["econ", "turista"], excluir=["prem"]),
    "premium": buscar_clase(["prem"]),
    "business": buscar_clase(["busi", "ejec"]),
    "primera": buscar_clase(["prim", "first"]),
}
if ids_clase["eco"] is None:                   # si no hay "económica", usa la primera fila como clase base
    ids_clase["eco"] = int(clase["id_clase"].iloc[0])
CLASES = {k: ((ids_clase[k] if ids_clase[k] is not None else ids_clase["eco"]), MULT[k] if ids_clase[k] is not None else 1.0)
          for k in MULT}
print("Clases detectadas (id, multiplicador):", CLASES)

# =====================================================================
# RESERVA, BILLETE, TARJETA_EMBARQUE, EQUIPAJE
# =====================================================================
muestra = df.dropna(subset=["capacidad_total", "dist_km"])
if NUM_VUELOS_MUESTRA:
    muestra = muestra.sample(n=min(NUM_VUELOS_MUESTRA, len(muestra)), random_state=SEMILLA)
muestra = muestra.sort_values("sal")

ids_pas = pasajero["id_pasajero"].to_numpy()
adulto = (pasajero["fecha_nacimiento"] <= pd.Timestamp(t0, unit="m") - pd.DateOffset(years=18)).to_numpy()
libre_pas = np.full(len(ids_pas), -10**12, dtype="int64")

ALFABETO = np.array(list("ABCDEFGHJKLMNPQRSTUVWXYZ23456789"))
localizadores = set()


def nuevo_localizador():
    while True:
        cod = "".join(rng.choice(ALFABETO, 6))
        if cod not in localizadores:
            localizadores.add(cod)
            return cod


R, B, T, E = [], [], [], []   # filas de reserva, billete, tarjeta, equipaje
bolsas_perdidas = []          # (id_vuelo, minuto de llegada) para crear incidencias
id_reserva = id_billete = 0

for v in muestra.itertuples(index=False):
    cap = int(v.capacidad_total)
    n = max(1, min(cap, int(round(cap * np.clip(rng.normal(OCUPACION_MEDIA, 0.08), 0.35, 1.0)))))
    cancelado_vuelo = v.estado == "cancelado"

    # --- pasajeros libres (sin vuelos solapados) ---
    elegidos = np.array([], dtype=int)
    tam = min(len(ids_pas), n * 3)
    while len(elegidos) < n and tam <= len(ids_pas):
        cand = rng.choice(len(ids_pas), size=tam, replace=False)
        elegidos = cand[libre_pas[cand] <= v.sal - 120][:n]
        if tam == len(ids_pas):
            break
        tam = min(len(ids_pas), tam * 2)
    n = len(elegidos)
    if n == 0:
        continue
    libre_pas[elegidos] = v.lleg_prev + 60

    # --- grupos de reserva: el titular siempre es adulto ---
    tamanos = []
    while sum(tamanos) < n:
        tamanos.append(int(rng.choice([1, 2, 3, 4], p=[0.5, 0.3, 0.12, 0.08])))
    ad, me = elegidos[adulto[elegidos]], elegidos[~adulto[elegidos]]
    m = min(len(tamanos), len(ad))
    if m == 0:
        continue
    resto = np.concatenate([ad[m:], me]); rng.shuffle(resto)
    grupos = [[ad[i]] for i in range(m)]
    for j, p in enumerate(resto):          # reparte al resto en bucle entre los grupos
        grupos[j % m].append(p)

    # --- asientos y clases ---
    abreast, letras = (10, "ABCDEFGHJK") if cap >= 300 else (8, "ABCDEFGH") if cap >= 230 else (6, "ABCDEF") if cap >= 130 else (4, "ABCD")
    asientos = [f"{r}{l}" for r in range(1, math.ceil(cap / abreast) + 1) for l in letras][:cap]
    pos = rng.choice(cap, size=n, replace=False)
    f_prim = 0.02 if cap >= 250 else 0.0
    f_bus = 0.07 if cap >= 100 else 0.0
    f_pre = 0.08 if cap >= 250 else 0.0
    pos_asiento = {}
    for g in grupos:
        for p in g:
            pos_asiento[p] = None
    orden = list(pos_asiento)
    for p, s in zip(orden, pos):
        pos_asiento[p] = s

    for g in grupos:
        id_reserva += 1
        fecha_res = v.sal - 180 - min(rng.exponential(30 * 1440), 330 * 1440)
        reserva_cancelada = cancelado_vuelo or rng.random() < PROB_RESERVA_CANCELADA
        importe = 0.0
        for p in g:
            s = pos_asiento[p]
            frac = s / cap
            clave = "primera" if frac < f_prim else "business" if frac < f_prim + f_bus else \
                    "premium" if frac < f_prim + f_bus + f_pre else "eco"
            id_cl, mult = CLASES[clave]
            precio = round(float(np.clip((TARIFA_BASE + TARIFA_KM * v.dist_km) * mult * rng.lognormal(0, 0.25), 20, 99999)), 2)
            if reserva_cancelada:
                est, asiento = "cancelado", None
            else:
                est, asiento = ("emitido" if rng.random() < PROB_NO_SHOW else "embarcado"), asientos[s]
                importe += precio
            id_billete += 1
            B.append((id_billete, id_reserva, int(ids_pas[p]), int(v.id_vuelo), id_cl, asiento, precio, est))

            if est == "embarcado":
                lo, hi = v.sal - 120, max(v.sal - 120, v.sal_real - 45)
                T.append((id_billete, rng.uniform(lo, hi), rng.uniform(v.sal_real - 40, v.sal_real - 15)))
                for _ in range(int(rng.choice([0, 1, 2, 3], p=[0.55, 0.35, 0.08, 0.02]))):
                    perdida = rng.random() < PROB_PERDIDA_MALETA
                    E.append((id_billete, round(float(np.clip(rng.normal(17, 5), 2, 32)), 2),
                              "perdido" if perdida else "en_cinta"))
                    if perdida:
                        bolsas_perdidas.append((int(v.id_vuelo), int(v.lleg)))
        R.append((id_reserva, nuevo_localizador(), int(ids_pas[g[0]]), fecha_res,
                  "cancelada" if reserva_cancelada else "confirmada", round(importe, 2)))

reserva = pd.DataFrame(R, columns=["id_reserva", "localizador", "id_pasajero_titular", "fecha_reserva", "estado", "importe_total"])
reserva["fecha_reserva"] = texto(reserva["fecha_reserva"])
billete = pd.DataFrame(B, columns=["id_billete", "id_reserva", "id_pasajero", "id_vuelo", "id_clase", "asiento", "precio", "estado"])
tarjeta = pd.DataFrame(T, columns=["id_billete", "fecha_checkin", "hora_embarque"])
tarjeta.insert(0, "id_embarque", tarjeta.index + 1)
tarjeta["fecha_checkin"] = texto(tarjeta["fecha_checkin"]); tarjeta["hora_embarque"] = texto(tarjeta["hora_embarque"])
equipaje = pd.DataFrame(E, columns=["id_billete", "peso_kg", "estado"])
equipaje.insert(0, "id_equipaje", equipaje.index + 1)

# =====================================================================
# MANTENIMIENTO (siempre en huecos en los que la aeronave no vuela)
# =====================================================================
TIPOS_MANT = [  # tipo, veces/año, duración min (h), duración máx (h), coste mín, coste máx (para 180 plazas)
    ("Revisión A", 5, 8, 20, 8_000, 25_000),
    ("Mantenimiento de línea", 6, 1, 4, 300, 4_000),
    ("Cambio de neumáticos y frenos", 2, 2, 5, 4_000, 12_000),
    ("Inspección boroscópica de motor", 1, 4, 10, 5_000, 15_000),
    ("Revisión C", 0.5, 7 * 24, 18 * 24, 150_000, 600_000),
]
anios = (t1 - t0) / (365 * 1440)
M = []
cap_aeronave = df.drop_duplicates("id_aeronave").set_index("id_aeronave")["capacidad_total"]
estado_aeronave = aeronave.set_index("id_aeronave")["estado"]

for id_a, g in df.groupby("id_aeronave"):
    ini = (g["sal"] - 60).to_numpy()
    fin = np.maximum(g["lleg"], g["lleg_prev"]).to_numpy() + 45
    o = np.argsort(ini); ini, fin = ini[o], fin[o]
    fin_acum = np.maximum.accumulate(fin)
    gap_ini = np.concatenate([[t0], fin_acum])
    gap_fin = np.concatenate([ini, [t1]])
    ok = gap_fin > gap_ini
    gap_ini, gap_fin = gap_ini[ok], gap_fin[ok]
    factor = float(np.nan_to_num(cap_aeronave.get(id_a, 180), nan=180)) / 180
    for tipo, por_anio, dmin, dmax, cmin, cmax in TIPOS_MANT:
        for _ in range(int(rng.poisson(por_anio * anios))):
            dur = int(rng.uniform(dmin, dmax) * 60)
            nominal = rng.uniform(t0, t1)
            i = int(np.searchsorted(gap_fin, nominal))
            for j in range(i, min(i + 200, len(gap_ini))):
                inicio = max(gap_ini[j], nominal) + 15
                if inicio + dur + 15 <= gap_fin[j]:
                    M.append((int(id_a), tipo, inicio, inicio + dur,
                              round(float(rng.uniform(cmin, cmax) * factor), 2)))
                    # el hueco usado se parte para no solaparse con otro mantenimiento
                    gap_ini = np.append(gap_ini, inicio + dur + 15); gap_fin = np.append(gap_fin, gap_fin[j])
                    gap_fin[j] = inicio - 15
                    o2 = np.argsort(gap_ini); gap_ini, gap_fin = gap_ini[o2], gap_fin[o2]
                    break
    if estado_aeronave.get(id_a) == "mantenimiento":   # aeronave actualmente en mantenimiento: orden abierta
        M.append((int(id_a), "Revisión C", int(fin_acum[-1]) + 60, np.nan, 0.0))

mantenimiento = pd.DataFrame(M, columns=["id_aeronave", "tipo", "fecha_inicio", "fecha_fin", "coste"])
mantenimiento = mantenimiento.sort_values(["fecha_inicio", "id_aeronave"]).reset_index(drop=True)
mantenimiento.insert(0, "id_mantenimiento", mantenimiento.index + 1)
mantenimiento["fecha_inicio"] = texto(mantenimiento["fecha_inicio"]); mantenimiento["fecha_fin"] = texto(mantenimiento["fecha_fin"])

# =====================================================================
# INCIDENCIA (retrasos y cancelaciones derivados de los vuelos reales + incidentes aleatorios)
# =====================================================================
I = []
retraso_llegada = df["lleg"] - df["lleg_prev"]
for r in df[(df["estado"] == "retrasado") & (retraso_llegada >= 60)].itertuples():
    x = int(r.lleg - r.lleg_prev)
    grav = "critica" if x >= 600 else "alta" if x >= 240 else "media" if x >= 120 else "baja"
    I.append((int(r.id_vuelo), "Retraso", grav, f"Llegada con {x} minutos de retraso sobre la hora prevista.", r.lleg))
for r in df[df["estado"] == "cancelado"].itertuples():
    I.append((int(r.id_vuelo), "Cancelación de vuelo", "media", "Vuelo cancelado antes de la salida.", r.sal - 60))

ALEATORIOS = {  # tipo: (peso, [(gravedad, prob)], descripción)
    "Impacto con aves": (20, {"baja": .55, "media": .3, "alta": .13, "critica": .02}, "Impacto con aves durante la fase de despegue o aproximación; inspección posterior."),
    "Fallo técnico": (25, {"baja": .4, "media": .4, "alta": .17, "critica": .03}, "Aviso de fallo en un sistema de la aeronave; revisión por mantenimiento."),
    "Emergencia médica a bordo": (18, {"baja": .3, "media": .5, "alta": .18, "critica": .02}, "Pasajero con emergencia médica atendido por la tripulación."),
    "Pasajero conflictivo": (15, {"baja": .6, "media": .35, "alta": .05}, "Incidente de conducta de un pasajero gestionado por la tripulación de cabina."),
    "Turbulencia severa": (12, {"baja": .4, "media": .45, "alta": .14, "critica": .01}, "Turbulencia fuerte en ruta con aviso de cinturones y revisión de cabina."),
    "Desvío a aeropuerto alternativo": (5, {"media": .6, "alta": .35, "critica": .05}, "Desvío a un aeropuerto alternativo por causa operativa."),
}
tipos = list(ALEATORIOS)
pesos_t = np.array([ALEATORIOS[t][0] for t in tipos], dtype=float); pesos_t /= pesos_t.sum()
ops = df[df["estado"] != "cancelado"]
sel = ops[rng.random(len(ops)) < PROB_INCIDENCIA_ALEATORIA]
for r in sel.itertuples():
    t = tipos[int(rng.choice(len(tipos), p=pesos_t))]
    _, gdist, desc = ALEATORIOS[t]
    grav = rng.choice(list(gdist), p=np.array(list(gdist.values())) / sum(gdist.values()))
    I.append((int(r.id_vuelo), t, str(grav), desc, rng.uniform(r.sal_real, max(r.sal_real + 1, r.lleg))))
perdidas = pd.DataFrame(bolsas_perdidas, columns=["id_vuelo", "lleg"]).groupby("id_vuelo").agg(n=("lleg", "size"), lleg=("lleg", "first"))
for id_v, r in perdidas.iterrows():   # una incidencia por vuelo con equipaje extraviado
    I.append((int(id_v), "Equipaje extraviado", "baja" if r.n < 5 else "media",
              f"{int(r.n)} maleta(s) facturada(s) no localizada(s) a la llegada.", r.lleg + rng.uniform(30, 240)))

incidencia = pd.DataFrame(I, columns=["id_vuelo", "tipo", "gravedad", "descripcion", "fecha_hora"])
incidencia = incidencia.sort_values(["fecha_hora", "id_vuelo"]).reset_index(drop=True)
incidencia.insert(0, "id_incidencia", incidencia.index + 1)
incidencia["fecha_hora"] = texto(incidencia["fecha_hora"])

# =====================================================================
# EXPORTAR
# =====================================================================
salidas = {"reserva": reserva, "billete": billete, "tarjeta_embarque": tarjeta,
           "equipaje": equipaje, "mantenimiento": mantenimiento, "incidencia": incidencia}
for nombre, d in salidas.items():
    d.to_csv(carpeta_data / f"{nombre}.csv", index=False)
    print(f"{nombre}.csv: {len(d)} filas")

print(f"\nVuelos con billetes: {muestra['id_vuelo'].nunique()} de {len(df)}")
print("Billetes por estado:\n" + billete["estado"].value_counts().to_string())
print("Incidencias por tipo:\n" + incidencia["tipo"].value_counts().to_string())

Clases detectadas (id, multiplicador): {'eco': (1, 1.0), 'premium': (2, 1.6), 'business': (3, 3.2), 'primera': (4, 5.0)}
reserva.csv: 375557 filas
billete.csv: 664853 filas
tarjeta_embarque.csv: 589071 filas
equipaje.csv: 335035 filas
mantenimiento.csv: 12273 filas
incidencia.csv: 12319 filas

Vuelos con billetes: 5000 de 114892
Billetes por estado:
estado
embarcado    589071
cancelado     51194
emitido       24588
Incidencias por tipo:
tipo
Retraso                            9224
Equipaje extraviado                1678
Cancelación de vuelo               1005
Fallo técnico                       100
Emergencia médica a bordo            91
Impacto con aves                     79
Pasajero conflictivo                 61
Turbulencia severa                   58
Desvío a aeropuerto alternativo      23


In [21]:
from pathlib import Path

import numpy as np
import pandas as pd

# 1. Definir rutas
carpeta_raw = Path("raw_data")
carpeta_data = Path("data")
carpeta_data.mkdir(parents=True, exist_ok=True)

ruta_flights_csv = carpeta_raw / "flights.csv"   # <- cambia el nombre si el de Kaggle es otro
ruta_aeropuerto_csv = carpeta_data / "aeropuerto.csv"
ruta_aerolinea_csv = carpeta_data / "aerolinea.csv"
ruta_ruta_csv = carpeta_data / "ruta.csv"
ruta_aeronave_csv = carpeta_data / "aeronave.csv"
ruta_terminal_csv = carpeta_data / "terminal.csv"
ruta_puerta_csv = carpeta_data / "puerta_embarque.csv"
ruta_pista_csv = carpeta_data / "pista.csv"
ruta_vuelo_csv = carpeta_data / "vuelo.csv"

for ruta in (ruta_flights_csv, ruta_aeropuerto_csv, ruta_aerolinea_csv, ruta_ruta_csv, ruta_aeronave_csv,
             ruta_terminal_csv, ruta_puerta_csv, ruta_pista_csv):
    if not ruta.exists():
        raise FileNotFoundError(f"Falta '{ruta}'.")

TZ_ORIGEN = "America/New_York"   # el dataset solo tiene salidas de EWR, JFK y LGA
UMBRAL_RETRASO_MIN = 15          # criterio estándar: llegada con 15 min o más de retraso
OCUPACION_PUERTA_MIN = 45        # una puerta queda reservada 45 min antes de cada salida
SEMILLA = 42

# 2. Leer vuelos
df = pd.read_csv(ruta_flights_csv, dtype={"carrier": str, "tailnum": str, "origin": str, "dest": str})
n_inicial = len(df)
df = df.dropna(subset=["carrier", "flight", "origin", "dest", "sched_dep_time", "sched_arr_time", "tailnum"])
df["tailnum"] = df["tailnum"].str.strip().str.upper()

# 3. Tablas ya limpias
df_aero = pd.read_csv(ruta_aeropuerto_csv, usecols=["id_aeropuerto", "codigo_iata", "zona_horaria"])
df_aerolinea = pd.read_csv(ruta_aerolinea_csv, usecols=["id_aerolinea", "codigo_iata"]).dropna()
df_ruta = pd.read_csv(ruta_ruta_csv)           # columnas esperadas: id_ruta, origen, destino (ids de aeropuerto)
df_aeronave = pd.read_csv(ruta_aeronave_csv)   # id_aeronave, matricula, id_modelo, id_aerolinea, estado

# 4. id_ruta: IATA origen/destino -> id_aeropuerto -> ruta
print("Columnas de ruta.csv:", list(df_ruta.columns))
col_origen = next(c for c in df_ruta.columns if "orig" in c.lower())
col_destino = next(c for c in df_ruta.columns if "dest" in c.lower())
col_id_ruta = next(c for c in df_ruta.columns if c.lower().startswith("id_ruta"))
print(f"Usando: {col_id_ruta}, {col_origen}, {col_destino}")

iata_a_id = df_aero.drop_duplicates("codigo_iata").set_index("codigo_iata")["id_aeropuerto"]
df["id_origen"] = df["origin"].map(iata_a_id)
df["id_destino"] = df["dest"].map(iata_a_id)
df = df.dropna(subset=["id_origen", "id_destino"])
df["id_origen"] = df["id_origen"].astype(int)
df["id_destino"] = df["id_destino"].astype(int)

df_ruta = df_ruta.rename(columns={col_id_ruta: "id_ruta", col_origen: "ruta_origen", col_destino: "ruta_destino"})
df = df.merge(
    df_ruta[["id_ruta", "ruta_origen", "ruta_destino"]],
    left_on=["id_origen", "id_destino"], right_on=["ruta_origen", "ruta_destino"], how="inner",
)
n_tras_ruta = len(df)
if df.empty:
    raise ValueError("Ningún vuelo cruza con ruta.csv: revisa que origen/destino de ruta sean id_aeropuerto.")

# 5. id_aeronave: por matrícula, y solo si la aeronave pertenece hoy a la misma aerolínea que operó el vuelo
df = df.merge(df_aeronave[["id_aeronave", "matricula", "id_aerolinea"]],
              left_on="tailnum", right_on="matricula", how="inner")
n_tras_matricula = len(df)
if df.empty:
    raise ValueError("Ningún tailnum coincide con aeronave.matricula (aviones de 2013 frente a la base de 2025).")
id_aerolinea_carrier = df_aerolinea.drop_duplicates("codigo_iata").set_index("codigo_iata")["id_aerolinea"]
df = df[df["id_aerolinea"] == df["carrier"].map(id_aerolinea_carrier)]
n_tras_aerolinea = len(df)
print(f"Con ruta: {n_tras_ruta} | con matrícula: {n_tras_matricula} | misma aerolínea: {n_tras_aerolinea}")
if df.empty:
    raise ValueError("Ninguna aeronave pertenece hoy a la aerolínea que operó el vuelo.")

# 6. numero_vuelo = carrier + número (ej. UA1545)
df["numero_vuelo"] = df["carrier"].str.strip().str.upper() + df["flight"].astype(int).astype(str)
df = df[df["numero_vuelo"].str.len() <= 10]

# 7. Horas programadas. Se guardan TODAS en UTC para que sean comparables entre zonas horarias.
fecha = pd.to_datetime(dict(year=df["year"], month=df["month"], day=df["day"]))


def minutos(hhmm):
    hhmm = hhmm.astype(int)
    return pd.to_timedelta((hhmm // 100) * 60 + hhmm % 100, unit="m")


salida_local = fecha + minutos(df["sched_dep_time"])
df["salida_prevista"] = (
    salida_local.dt.tz_localize(TZ_ORIGEN, ambiguous="NaT", nonexistent="shift_forward").dt.tz_convert("UTC")
)

# sched_arr_time está en hora LOCAL del aeropuerto de destino
zona_destino = df["id_destino"].astype(int).map(df_aero.set_index("id_aeropuerto")["zona_horaria"])
llegada_local = fecha + minutos(df["sched_arr_time"])
llegada_utc = pd.Series(pd.NaT, index=df.index, dtype="datetime64[ns, UTC]")
for zona, idx in zona_destino.groupby(zona_destino).groups.items():
    llegada_utc.loc[idx] = (
        llegada_local.loc[idx].dt.tz_localize(zona, ambiguous="NaT", nonexistent="shift_forward").dt.tz_convert("UTC")
    )
df["llegada_prevista"] = llegada_utc
df = df.dropna(subset=["salida_prevista", "llegada_prevista"]).copy()

# Si la llegada cae antes que la salida (vuelo que cruza medianoche), suma días
for _ in range(2):
    mal = df["llegada_prevista"] <= df["salida_prevista"]
    df.loc[mal, "llegada_prevista"] += pd.Timedelta(days=1)
df = df[df["llegada_prevista"] > df["salida_prevista"]]

# 8. Horas reales = prevista + retraso. Sin dep_time o sin arr_time: cancelado (o desviado) y sin horas reales
df["salida_real"] = df["salida_prevista"] + pd.to_timedelta(df["dep_delay"], unit="m")
df["llegada_real"] = df["llegada_prevista"] + pd.to_timedelta(df["arr_delay"], unit="m")
sin_datos = df["dep_time"].isna() | df["arr_time"].isna() | df["arr_delay"].isna()
df.loc[sin_datos, ["salida_real", "llegada_real"]] = pd.NaT

# 9. Estado
df["estado"] = np.where(
    sin_datos, "cancelado",
    np.where(df["arr_delay"] >= UMBRAL_RETRASO_MIN, "retrasado", "aterrizado"),
)

# 11. Quitar duplicados de la UNIQUE (numero_vuelo, salida_prevista) y crear id_vuelo
df = df.drop_duplicates(subset=["numero_vuelo", "salida_prevista"])
df = df.sort_values("salida_prevista").reset_index(drop=True)
df["id_vuelo"] = df.index + 1

# 12. id_puerta e id_pista (de SALIDA) a partir de puerta_embarque.csv y pista.csv.
#     Solo hay puerta/pista si el aeropuerto de origen existe en esos CSV; si no, queda NULL.
#     La elección entre las disponibles del aeropuerto es ALEATORIA (sintética), no el dato real.
rng = np.random.default_rng(SEMILLA)
df_terminal = pd.read_csv(ruta_terminal_csv, usecols=["id_terminal", "id_aeropuerto"])
df_puerta = pd.read_csv(ruta_puerta_csv, usecols=["id_puerta", "id_terminal"]).merge(df_terminal, on="id_terminal")
df_pista = pd.read_csv(ruta_pista_csv, usecols=["id_pista", "id_aeropuerto", "longitud_m"])

puertas_por_aero = {a: g["id_puerta"].to_numpy() for a, g in df_puerta.groupby("id_aeropuerto")}
pistas_por_aero = {
    a: (g["id_pista"].to_numpy(), (g["longitud_m"] / g["longitud_m"].sum()).to_numpy())  # más peso a pistas largas
    for a, g in df_pista.groupby("id_aeropuerto")
}

salida_min = ((df["salida_prevista"] - pd.Timestamp("1970-01-01", tz="UTC")) // pd.Timedelta(minutes=1)).to_numpy()  # minutos desde epoch (UTC)
origen_ids = df["id_origen"].to_numpy()
id_puerta = np.full(len(df), np.nan)
id_pista = np.full(len(df), np.nan)
libre_desde = {}  # id_puerta -> minuto a partir del cual queda libre

for i in range(len(df)):  # df ya está ordenado por salida_prevista
    ao = origen_ids[i]
    if ao in puertas_por_aero:
        libres = [g for g in puertas_por_aero[ao] if libre_desde.get(g, -1) <= salida_min[i] - OCUPACION_PUERTA_MIN]
        if libres:
            g = rng.choice(libres)
            id_puerta[i] = g
            libre_desde[g] = salida_min[i]
    if ao in pistas_por_aero:
        ids, pesos = pistas_por_aero[ao]
        id_pista[i] = rng.choice(ids, p=pesos)

df["id_puerta"] = pd.Series(id_puerta, index=df.index).astype("Int64")
df["id_pista"] = pd.Series(id_pista, index=df.index).astype("Int64")



def fmt(col):
    return col.dt.tz_convert("UTC").dt.strftime("%Y-%m-%d %H:%M:%S")


for c in ("salida_prevista", "llegada_prevista", "salida_real", "llegada_real"):
    df[c] = fmt(df[c])

columnas_finales = [
    "id_vuelo", "numero_vuelo", "id_ruta", "id_aeronave", "id_puerta", "id_pista",
    "salida_prevista", "llegada_prevista", "salida_real", "llegada_real", "estado",
]
df_final = df[columnas_finales]

# 12. Exportar (los NULL salen como celda vacía; en LOAD DATA usa NULLIF o SET para convertirlos)
df_final.to_csv(ruta_vuelo_csv, index=False)

print(f"Vuelos leídos: {n_inicial}")
print(f"Con ruta en tu tabla ruta: {n_tras_ruta}")
print(f"Con matrícula en aeronave: {n_tras_matricula}")
print(f"Con la misma aerolínea que hoy: {n_tras_aerolinea}")
print(f"Vuelos finales: {len(df_final)}")
print(f"Con id_puerta: {df_final['id_puerta'].notna().sum()} | con id_pista: {df_final['id_pista'].notna().sum()}")
print(df_final["estado"].value_counts().to_string())
print(f"Archivo guardado en {ruta_vuelo_csv}")
df_final.head(10)

Columnas de ruta.csv: ['id_ruta', 'id_aeropuerto_origen', 'id_aeropuerto_destino', 'distancia_km']
Usando: id_ruta, id_aeropuerto_origen, id_aeropuerto_destino
Con ruta: 331152 | con matrícula: 138561 | misma aerolínea: 114892
Vuelos leídos: 336776
Con ruta en tu tabla ruta: 331152
Con matrícula en aeronave: 138561
Con la misma aerolínea que hoy: 114892
Vuelos finales: 114892
Con id_puerta: 35868 | con id_pista: 114892
estado
aterrizado    87068
retrasado     26819
cancelado      1005
Archivo guardado en data\vuelo.csv


,id_vuelo,numero_vuelo,id_ruta,id_aeronave,id_puerta,id_pista,salida_prevista,llegada_prevista,salida_real,llegada_real,estado
0,1,UA1545,33837,14605,134,240934,2013-01-01 10:15:00,2013-01-01 14:19:00,2013-01-01 10:17:00,2013-01-01 14:30:00,aterrizado
1,2,UA1714,15850,14702,<NA>,244204,2013-01-01 10:29:00,2013-01-01 14:30:00,2013-01-01 10:33:00,2013-01-01 14:50:00,retrasado
2,3,B6725,11464,8701,<NA>,242385,2013-01-01 10:45:00,2013-01-01 14:22:00,2013-01-01 10:44:00,2013-01-01 14:04:00,aterrizado
3,4,B61806,5119,8684,<NA>,242384,2013-01-01 10:59:00,2013-01-01 12:06:00,2013-01-01 10:59:00,2013-01-01 12:02:00,aterrizado
4,5,B6343,11414,8664,209,240934,2013-01-01 11:00:00,2013-01-01 13:50:00,2013-01-01 11:01:00,2013-01-01 13:44:00,aterrizado
5,6,B6371,11511,8633,<NA>,244204,2013-01-01 11:00:00,2013-01-01 13:58:00,2013-01-01 11:00:00,2013-01-01 13:51:00,aterrizado
6,7,UA1187,33841,14982,202,240933,2013-01-01 11:00:00,2013-01-01 17:02:00,2013-01-01 10:59:00,2013-01-01 16:54:00,aterrizado
7,8,B6125,11472,8645,<NA>,242384,2013-01-01 11:00:00,2013-01-01 14:01:00,NaN,NaN,cancelado
8,9,B649,11485,8698,<NA>,242384,2013-01-01 11:00:00,2013-01-01 13:51:00,2013-01-01 10:58:00,2013-01-01 13:49:00,aterrizado
9,10,B679,5147,8631,<NA>,242385,2013-01-01 11:00:00,2013-01-01 13:46:00,2013-01-01 10:57:00,2013-01-01 13:38:00,aterrizado


In [22]:
from pathlib import Path
import heapq
import math

import numpy as np
import pandas as pd

# 1. Definir rutas
carpeta_data = Path("data")
ruta_vuelo_csv = carpeta_data / "vuelo.csv"
ruta_aeronave_csv = carpeta_data / "aeronave.csv"
ruta_modelo_csv = carpeta_data / "modelo_aeronave.csv"
ruta_empleado_csv = carpeta_data / "empleado.csv"
ruta_tripulacion_csv = carpeta_data / "tripulacion_vuelo.csv"

for ruta in (ruta_vuelo_csv, ruta_aeronave_csv, ruta_modelo_csv, ruta_empleado_csv):
    if not ruta.exists():
        raise FileNotFoundError(f"Falta '{ruta}'. Genéralo antes de asignar tripulaciones.")

# 2. Parámetros (SINTÉTICOS salvo la regla de dotación de TCP)
PLAZAS_POR_TCP = 50          # 1 TCP por cada 50 plazas (regla de dotación mínima; contrástala antes de citarla)
MIN_ANTES_SALIDA = 60        # la tripulación se presenta 60 min antes de la salida prevista
MIN_DESCANSO_TRAS_VUELO = 45 # tiempo mínimo entre el fin de un vuelo y el siguiente del mismo empleado
SEMILLA = 42

rng = np.random.default_rng(SEMILLA)

# 3. Leer datos
vuelo = pd.read_csv(
    ruta_vuelo_csv,
    usecols=["id_vuelo", "id_aeronave", "salida_prevista", "llegada_prevista", "llegada_real", "estado"],
    parse_dates=["salida_prevista", "llegada_prevista", "llegada_real"],
)
aeronave = pd.read_csv(ruta_aeronave_csv, usecols=["id_aeronave", "id_modelo", "id_aerolinea"])
modelo = pd.read_csv(ruta_modelo_csv)
if "id_modelo" not in modelo.columns:
    modelo["id_modelo"] = modelo.index + 1   # igual que AUTO_INCREMENT al cargar el CSV entero
empleado = pd.read_csv(ruta_empleado_csv, usecols=["id_empleado", "rol", "id_aerolinea"])

df = (
    vuelo.merge(aeronave, on="id_aeronave", how="left")
    .merge(modelo[["id_modelo", "capacidad_total"]], on="id_modelo", how="left")
)
df = df[df["estado"] != "cancelado"].sort_values("salida_prevista").reset_index(drop=True)  # un vuelo cancelado no se tripula


def a_minutos(serie):
    return ((serie - pd.Timestamp("1970-01-01")) // pd.Timedelta(minutes=1)).astype("int64")


salida = a_minutos(df["salida_prevista"]).to_numpy()
fin_vuelo = a_minutos(df["llegada_real"].fillna(df["llegada_prevista"])).to_numpy()

# 4. Un montón (heap) por (aerolínea, rol): el empleado libre desde hace más tiempo sale primero.
#    Elemento: (minuto en que queda libre, desempate aleatorio, id_empleado)
heaps = {}
for (aerolinea, rol), g in empleado.groupby(["id_aerolinea", "rol"]):
    ids = g["id_empleado"].to_numpy()
    items = [(-10**12, float(rng.random()), int(i)) for i in ids]
    heapq.heapify(items)
    heaps[(int(aerolinea), rol)] = items


def tomar(aerolinea, rol, cantidad, inicio):
    """Saca hasta 'cantidad' empleados libres a la hora 'inicio'."""
    heap = heaps.get((aerolinea, rol))
    elegidos = []
    while heap and len(elegidos) < cantidad and heap[0][0] <= inicio:
        elegidos.append(heapq.heappop(heap))
    return elegidos


def devolver(aerolinea, rol, elegidos, libre_desde):
    heap = heaps[(aerolinea, rol)]
    for _, desempate, emp in elegidos:
        heapq.heappush(heap, (libre_desde, desempate, emp))


# 5. Asignar tripulación vuelo a vuelo, en orden cronológico
filas = []
faltan = {"comandante": 0, "copiloto": 0, "tcp": 0}
sin_modelo = 0

for i, r in enumerate(df.itertuples(index=False)):
    if pd.isna(r.id_aerolinea) or pd.isna(r.capacidad_total):
        sin_modelo += 1
        continue
    aerolinea = int(r.id_aerolinea)
    inicio = salida[i] - MIN_ANTES_SALIDA
    libre_desde = fin_vuelo[i] + MIN_DESCANSO_TRAS_VUELO
    n_tcp = max(2, math.ceil(r.capacidad_total / PLAZAS_POR_TCP))  # incluye al jefe de cabina

    for rol, cantidad in (("piloto", 1), ("copiloto", 1), ("tcp", n_tcp)):
        elegidos = tomar(aerolinea, rol, cantidad, inicio)
        for k, (_, _, emp) in enumerate(elegidos):
            if rol == "piloto":
                funcion = "comandante"
            elif rol == "copiloto":
                funcion = "copiloto"
            else:
                funcion = "jefe_cabina" if k == 0 else "tcp"
            filas.append((int(r.id_vuelo), emp, funcion))
        if len(elegidos) < cantidad:
            faltan["comandante" if rol == "piloto" else rol] += cantidad - len(elegidos)
        if elegidos:
            devolver(aerolinea, rol, elegidos, libre_desde)

# 6. Columnas finales, iguales al CREATE TABLE
df_final = pd.DataFrame(filas, columns=["id_vuelo", "id_empleado", "funcion"])
df_final = df_final.drop_duplicates(subset=["id_vuelo", "id_empleado"]).sort_values(["id_vuelo", "id_empleado"])

# 7. Exportar
df_final.to_csv(ruta_tripulacion_csv, index=False)

print(f"Vuelos tripulados (sin cancelados): {len(df)} | sin aerolínea o modelo: {sin_modelo}")
print(f"Filas generadas: {len(df_final)}")
print(df_final["funcion"].value_counts().to_string())
print(f"Plazas SIN cubrir por falta de empleados libres: {faltan}")
if any(faltan.values()):
    print("-> Sube EMPLEADOS_POR_AERONAVE en generar_empleado.py y vuelve a generar empleados y tripulación.")
print(f"Archivo guardado en {ruta_tripulacion_csv}")
df_final.head(10)

Vuelos tripulados (sin cancelados): 113887 | sin aerolínea o modelo: 0
Filas generadas: 629145
funcion
tcp            287484
comandante     113887
copiloto       113887
jefe_cabina    113887
Plazas SIN cubrir por falta de empleados libres: {'comandante': 0, 'copiloto': 0, 'tcp': 0}
Archivo guardado en data\tripulacion_vuelo.csv


,id_vuelo,id_empleado,funcion
0,1,12508,comandante
1,1,13579,copiloto
5,1,14403,tcp
4,1,15090,tcp
2,1,15394,jefe_cabina
3,1,17220,tcp
6,2,12353,comandante
7,2,14290,copiloto
8,2,14894,jefe_cabina
11,2,16103,tcp
